# Seen and Unseen — Colab run: Exp 3 capacity pilot (background) + Exp 2 Kuka cache and certificate

**One configuration cell at the top. After any disconnect or restart: Runtime → Run all.** Never run a middle cell alone after a restart; every step is resumable, so Run all only redoes what is missing.

| Part | What | Where it runs | Download |
|---|---|---|---|
| A | Exp 3 effect-blind capacity pilot: ConvLSTM 16/64/256 channels, 72 blinded blocks, resumable from a blinded checkpoint on Drive | background process, started in cell A3 | `exp3_capacity_pilot_results.zip` (cell A5) |
| B | Exp 2: Kuka feature cache on Drive, then manifest → certificate → support → dry run | foreground, while A runs | `exp2_prep_results.zip` (cell B5) |

GPU runtime (A100 or L4), High-RAM on. Expected: Part B about 1–1.5 h; Part A about 2–3 h in the background; both zips download when ready. The Part B training cells at the end stay locked until a signed freeze exists.

In [ ]:
#@title 0. Configuration (run first; the only cell that defines settings)
# --- Part A: Exp 3 capacity pilot
TUNE_BLOCKS = 4                     #@param {type:"integer"}
TUNE_CHECKPOINTS = "300,600,1200"   #@param {type:"string"}
HIDDEN = "16,64,256"                #@param {type:"string"}
RUN_TUNING = True                   #@param {type:"boolean"}
RUN_PILOT = True                    #@param {type:"boolean"}
EXP3_DRIVE_DIR = "/content/drive/MyDrive/seen-and-unseen/exp3/capacity_pilot_v1"  #@param {type:"string"}
# --- Part B: Exp 2 cache + certificate
SHARDS = "0:1024:3"                 #@param {type:"string"}
PER_SHARD = 40                      #@param {type:"integer"}
WORKERS = 6                         #@param {type:"integer"}
DRYRUN_DATASETS = 300               #@param {type:"integer"}
EXP2_DRIVE_DIR = "/content/drive/MyDrive/seen-and-unseen/exp2"  #@param {type:"string"}
USE_DRIVE = True

import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch to a GPU runtime)", "| CPUs:", os.cpu_count())
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{EXP3_DRIVE_DIR}/DO_NOT_OPEN", exist_ok=True)
os.makedirs(f"{EXP2_DRIVE_DIR}/kuka", exist_ok=True); os.makedirs(f"{EXP2_DRIVE_DIR}/registered_out", exist_ok=True)
EXP3_CKPT = f"{EXP3_DRIVE_DIR}/blinded_checkpoint.jsonl"
EXP3_KEY = f"{EXP3_DRIVE_DIR}/DO_NOT_OPEN/blinding_key.json"
EXP3_LOG = f"{EXP3_DRIVE_DIR}/pilot.log"
OUT = f"{EXP2_DRIVE_DIR}/registered_out"
print("Exp 3 checkpoint:", EXP3_CKPT, "| blocks already recorded:", sum(1 for _ in open(EXP3_CKPT)) if os.path.exists(EXP3_CKPT) else 0)
print("Exp 2 outputs:", OUT)

## Part A — Experiment 3: effect-blind capacity pilot (background)

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title A1. Code: public trajectory-lineage at 6b1031b10ac5ec757e8bd8e7f1a2b47f0ddcbfef (includes pull request #19)
%cd /content
import os
if not os.path.exists("trajectory-lineage"):
    !git clone -q https://github.com/physicalreasoning/trajectory-lineage.git
%cd /content/trajectory-lineage
!git checkout -q 6b1031b10ac5ec757e8bd8e7f1a2b47f0ddcbfef && git rev-parse --short HEAD
!pip install -q numpy h5py pytest
print("ok")

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title A2. Run the capacity tests (about 1 min), effect-blind base tuning (base models only)
%cd /content/trajectory-lineage
!git log --oneline -1
!PYTHONPATH=src python -m pytest -q tests/test_capacity.py tests/test_analyze_2d.py tests/test_exp3_2d_path.py -p no:cacheprovider 2>&1 | tail -n 2
if RUN_TUNING and not os.path.exists(f"{EXP3_DRIVE_DIR}/exposure2d_capacity_base_tuning.json"):
    !PYTHONPATH=src python scripts/tune_base_2d.py --config experiments/exposure2d_capacity_pilot_v1.json --out results/exposure2d_capacity_base_tuning.json --blocks {TUNE_BLOCKS} --checkpoints {TUNE_CHECKPOINTS} --hidden {HIDDEN} 2>&1 | tail -n 12
    !cp results/exposure2d_capacity_base_tuning.json {EXP3_DRIVE_DIR}/
else:
    print("tuning skipped (disabled or already on Drive)")

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title A3. Launch the 72-block pilot in the background (resumes from the Drive checkpoint)
import subprocess, os, time
if RUN_PILOT:
    env = dict(os.environ, PYTHONPATH="/content/trajectory-lineage/src", TRAJECTORY_LINEAGE_WORKERS=str(max(2, (os.cpu_count() or 4) - 2)))
    cmd = ["python", "scripts/exposure_experiment.py", "--config", "experiments/exposure2d_capacity_pilot_v1.json",
           "--out", "results/exposure2d_capacity_pilot_v1_blinded.json", "--device", "cuda",
           "--blinded-checkpoint", EXP3_CKPT, "--blinding-key", EXP3_KEY]
    PILOT = subprocess.Popen(cmd, cwd="/content/trajectory-lineage", env=env, stdout=open(EXP3_LOG, "a"), stderr=subprocess.STDOUT)
    time.sleep(20)
    print("pilot PID", PILOT.pid, "| alive:", PILOT.poll() is None, "| log:", EXP3_LOG)
    !tail -n 3 {EXP3_LOG}
else:
    print("pilot disabled")

## Part B — Experiment 2: Kuka cache, manifest, certificate, support, dry run (runs while Part A works)

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title B1. Owner's pipeline and dependencies
%cd /content
import os
if not os.path.exists("rwmo"):
    !git clone -q https://github.com/subhashjprasad/robot-world-model-optimism-research.git rwmo
%cd /content/rwmo
!git rev-parse --short HEAD
!pip install -q tfrecord crc32c protobuf requests "transformers>=5.0" accelerate pillow scikit-learn scipy pandas matplotlib
!sed -i 's/device = "mps" if torch.backends.mps.is_available() else "cpu"/device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")/' experiment/extract_kuka.py
!sed -i 's/DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"/DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")/' experiment/wm.py
!rm -rf /content/rwmo/data/kuka && ln -s {EXP2_DRIVE_DIR}/kuka /content/rwmo/data/kuka
print("cache:", os.path.realpath("/content/rwmo/data/kuka"), "| shards present:", len(os.listdir("/content/rwmo/data/kuka")))

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title B2. Feature extraction (resumable; skips shards already in the cache)
%cd /content/rwmo
!PYTHONPATH=experiment:audit python experiment/extract_kuka.py --shards {SHARDS} --per-shard {PER_SHARD} --workers {WORKERS} 2>&1 | grep -v -i "warn" | tail -n 25
!ls data/kuka | wc -l; du -sh data/kuka

In [ ]:
#@title B3. Create the package folders (the next cells write the exp2-kuka-matched files)
import os
os.makedirs('/content/exp2-kuka-matched/configs', exist_ok=True)
os.makedirs('/content/exp2-kuka-matched/registered', exist_ok=True)
print('ok')

In [ ]:
%%writefile /content/exp2-kuka-matched/configs/exp2_registered.json
{
  "name": "exp2_transition_matched_kuka",
  "version": "2026-10-28",
  "description": "Registered construction of Experiment 2 (transition-matched Kuka control) for Seen and Unseen. Every value here is frozen before training.",
  "seeds": [
    100,
    101,
    102,
    103,
    104,
    105,
    106,
    107,
    108,
    109
  ],
  "historical_seeds_excluded": [
    0,
    1,
    2
  ],
  "optimizer_steps": 2000,
  "episode_eligibility": {
    "length_min": 3,
    "length_max": 24
  },
  "split": {
    "rule": "(shard // 3) % 5 == 0 is test",
    "divisor": 3,
    "every": 5
  },
  "designs": {
    "O": {
      "description": "Original episode-count arms (prefixes of the seeded pools).",
      "arms": {
        "phi0.10": {
          "n_success": 2700,
          "n_failure": 300
        },
        "phi0.65": {
          "n_success": 1050,
          "n_failure": 1950
        }
      }
    },
    "A": {
      "description": "Exactly `transitions` distinct transitions at each episode failure fraction phi; episode counts free, total episodes a multiple of `episode_multiple` so phi*N is an integer.",
      "transitions": 31600,
      "phis": [
        0.0,
        0.1,
        0.25,
        0.5,
        0.65
      ],
      "episode_multiple": 20,
      "max_candidate_totals": 8
    },
    "B": {
      "description": "Equal normalized outcome-conditional length distributions at a fixed transition budget. Each length contributes 20*k_l episodes: 18S+2F at phi=.10 and 7S+13F at phi=.65. The common histogram is the integer solution closest to the pooled availability shape (least squares) under the per-length caps.",
      "transitions": 31600,
      "phis": [
        0.1,
        0.65
      ],
      "unit": 20,
      "fallback": {
        "rule": "If 31,600 is infeasible, budget_B = largest multiple of 1000 <= 0.9 * (maximum feasible B budget). Design A comparators at budget_B are added for both B fractions.",
        "fraction_of_max": 0.9,
        "round_to": 1000
      }
    },
    "V": {
      "description": "Shared seeded base of `base_successes` successes; add distinct transitions of further successes (V+S) or of failures (V+F) until the arm holds exactly `total_transitions` (Design A's budget), so V arms are transition-matched to A.",
      "base_successes": 1500,
      "total_transitions": 31600
    }
  },
  "evaluation": {
    "min_eval_per_outcome_per_length": 20,
    "min_train_per_outcome_per_length": 1,
    "threshold": 0.5,
    "prevalence_source": "all eligible test episodes (before support exclusion)"
  },
  "decision": {
    "margin_points": 3.0,
    "alpha_support_one_sided": 0.05,
    "equivalence_ci_level": 0.9,
    "holm_alpha": 0.05,
    "primary": "A",
    "secondary": [
      "O",
      "B",
      "V",
      "O_minus_A",
      "A_minus_B"
    ]
  },
  "judge": {
    "per_class": 2500,
    "rng_seed": 99,
    "C": 0.05,
    "validity_auroc": 0.75,
    "pca_components": 256
  }
}


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/__init__.py
"""Registered construction, weighting, and analysis for Experiment 2 (transition-matched Kuka control).

Modules
-------
manifest   episode manifest (id, shard, length, outcome, split) from the cached npz shards; synthetic manifest for dry runs
design     seeded samplers for designs O, A, B, V and the feasibility certificate
weights    frozen evaluation support and overlap weights
analyze    the registered decision contract (paired t, TOST, Holm, missing-run handling)
dryrun     synthetic operating characteristics of the frozen analyzer
freeze     freeze manifest that binds code, config, manifest, selection, support and sign-offs
run_registered  training driver that consumes the frozen selection and writes per-episode scores

Only numpy and scipy are required for everything except ``run_registered`` (torch + the owner's ``wm.py``).
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any

PACKAGE_ROOT = Path(__file__).resolve().parents[1]
DEFAULT_CONFIG = PACKAGE_ROOT / "configs" / "exp2_registered.json"


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path) -> str:
    return sha256_bytes(Path(path).read_bytes())


def canonical_json(obj: Any) -> str:
    """Deterministic JSON (sorted keys, fixed separators) so hashes are reproducible."""

    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def sha256_json(obj: Any) -> str:
    return sha256_bytes(canonical_json(obj).encode("utf-8"))


def load_config(path: Path | str | None = None) -> dict:
    return json.loads(Path(path or DEFAULT_CONFIG).read_text())


def _jsonable(o: Any) -> Any:
    """Convert numpy scalars and arrays so json can serialize them."""

    import numpy as np

    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"Object of type {type(o).__name__} is not JSON serializable")


def write_json(path: Path, obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=1, sort_keys=False, default=_jsonable) + "\n")


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/manifest.py
"""Episode manifest: the metadata-only view of the Kuka cache that every registered step is built from.

An episode is identified by ``"<shard:05d>/<index>"`` (its position in the shard's npz arrays). The
manifest records length (frames), transitions (frames - 1), recorded success, shard, split and the
proprioception flag. Building it reads only the small ``length``/``success``/``proprio_ok`` arrays, so
it takes seconds and needs no features in memory.

Two constructors:

* ``build_manifest(npz_dir)``   from the real cache ``data/kuka/shard_*.npz`` (owner's machine);
* ``synthetic_manifest(...)``   a stand-in with the real pool sizes and the audited length-by-outcome
  shape, used only for dry runs and for checking the samplers at scale. It is labelled as synthetic
  in its header and must never be used to train or to sign a freeze.
"""

from __future__ import annotations

import argparse
import glob
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from . import sha256_bytes, write_json

LENGTH_MIN, LENGTH_MAX = 3, 24


def is_test_shard(shard: np.ndarray | int, divisor: int = 3, every: int = 5) -> np.ndarray | bool:
    """The owner's split rule: test = every 5th sampled shard (``shard_split`` in ``wm.py``)."""

    return (np.asarray(shard) // divisor) % every == 0


@dataclass
class Manifest:
    ids: np.ndarray  # str
    shard: np.ndarray  # int
    length: np.ndarray  # int frames
    success: np.ndarray  # int 0/1
    split: np.ndarray  # "train" | "test"
    proprio_ok: np.ndarray  # bool
    header: dict

    @property
    def transitions(self) -> np.ndarray:
        return self.length - 1

    def __len__(self) -> int:
        return len(self.ids)

    def eligible(self, lo: int = LENGTH_MIN, hi: int = LENGTH_MAX) -> np.ndarray:
        return (self.length >= lo) & (self.length <= hi)

    def train_mask(self) -> np.ndarray:
        return (self.split == "train") & self.eligible()

    def test_mask(self) -> np.ndarray:
        return (self.split == "test") & self.eligible()

    def index_of(self) -> dict[str, int]:
        return {i: k for k, i in enumerate(self.ids.tolist())}

    def to_rows(self) -> list[dict]:
        return [
            {"id": i, "shard": int(s), "length": int(l), "transitions": int(l) - 1, "success": int(y),
             "split": sp, "proprio_ok": bool(p)}
            for i, s, l, y, sp, p in zip(self.ids, self.shard, self.length, self.success, self.split, self.proprio_ok)
        ]

    def summary(self) -> dict:
        tr, te = self.train_mask(), self.test_mask()
        return {
            "episodes": int(len(self)),
            "eligible": int(self.eligible().sum()),
            "train": {"episodes": int(tr.sum()), "successes": int(self.success[tr].sum()),
                      "failures": int((1 - self.success[tr]).sum()),
                      "transitions": int(self.transitions[tr].sum())},
            "test": {"episodes": int(te.sum()), "successes": int(self.success[te].sum()),
                     "prevalence": float(self.success[te].mean()) if te.any() else None},
            "length_range": [int(self.length.min()), int(self.length.max())] if len(self) else None,
        }


def _canonical_bytes(header: dict, rows: list[dict]) -> bytes:
    lines = [json.dumps({"header": header}, sort_keys=True, separators=(",", ":"))]
    lines += [json.dumps(r, sort_keys=True, separators=(",", ":")) for r in rows]
    return ("\n".join(lines) + "\n").encode("utf-8")


def save_manifest(m: Manifest, path: Path) -> str:
    """Write JSONL (header line first, one episode per line). Returns the SHA-256 of the bytes."""

    data = _canonical_bytes(m.header, m.to_rows())
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return sha256_bytes(data)


def load_manifest(path: Path) -> tuple[Manifest, str]:
    data = Path(path).read_bytes()
    lines = data.decode("utf-8").splitlines()
    header = json.loads(lines[0])["header"]
    rows = [json.loads(l) for l in lines[1:] if l.strip()]
    m = Manifest(
        ids=np.array([r["id"] for r in rows], dtype=object),
        shard=np.array([r["shard"] for r in rows], dtype=int),
        length=np.array([r["length"] for r in rows], dtype=int),
        success=np.array([r["success"] for r in rows], dtype=int),
        split=np.array([r["split"] for r in rows], dtype=object),
        proprio_ok=np.array([r["proprio_ok"] for r in rows], dtype=bool),
        header=header,
    )
    return m, sha256_bytes(data)


def build_manifest(npz_dir: Path, divisor: int = 3, every: int = 5) -> Manifest:
    """Read lengths and labels from every ``shard_*.npz`` (features are not loaded)."""

    files = sorted(glob.glob(str(Path(npz_dir) / "shard_*.npz")))
    if not files:
        raise FileNotFoundError(f"no shard_*.npz under {npz_dir}")
    ids, shards, lengths, succ, prop = [], [], [], [], []
    for f in files:
        sid = int(Path(f).stem.split("_")[1])
        with np.load(f) as npz:
            ln, sc = npz["length"], npz["success"]
            ok = npz["proprio_ok"] if "proprio_ok" in npz.files else np.ones(len(ln), bool)
        for e in range(len(ln)):
            ids.append(f"{sid:05d}/{e}")
            shards.append(sid)
            lengths.append(int(ln[e]))
            succ.append(int(sc[e]))
            prop.append(bool(ok[e]))
    shard = np.array(shards, int)
    split = np.where(is_test_shard(shard, divisor, every), "test", "train").astype(object)
    header = {"kind": "real", "source": str(npz_dir), "shards": len(files), "split_rule": f"(shard // {divisor}) % {every} == 0 is test",
              "eligibility": {"length_min": LENGTH_MIN, "length_max": LENGTH_MAX}}
    return Manifest(np.array(ids, dtype=object), shard, np.array(lengths, int), np.array(succ, int), split, np.array(prop, bool), header)


# ---------------------------------------------------------------- synthetic stand-in

def audited_length_table(audit_jsonl: Path) -> dict[int, np.ndarray]:
    """Length counts by outcome from the owner's audit sample (``results/episodes/kuka.jsonl``).

    Returns {0: counts over lengths 3..24, 1: counts}. Laplace smoothing (+0.25) keeps every
    eligible length possible, so the synthetic pool is not artificially infeasible."""

    rows = [json.loads(l) for l in Path(audit_jsonl).read_text().splitlines() if l.strip()]
    table = {y: np.full(LENGTH_MAX - LENGTH_MIN + 1, 0.25) for y in (0, 1)}
    for r in rows:
        L = int(r["n_steps"])
        if LENGTH_MIN <= L <= LENGTH_MAX:
            table[int(r["success"])][L - LENGTH_MIN] += 1
    return table


def synthetic_manifest(audit_jsonl: Path, *, train_successes: int = 3989, train_failures: int = 6900,
                       test_successes: int = 996, test_failures: int = 1804, seed: int = 20261023) -> Manifest:
    """A metadata-only stand-in with the real pool sizes and the audited length shape.

    Pool sizes: the owner's log shows 3,989 train successes and 2,800 test episodes with 996
    successes; the 3,000 train failures in that log were a memory cap, and the README reports
    13.7k extracted episodes, so about 6,900 train failures exist. The failure count is an
    assumption and is stated in the header."""

    rng = np.random.default_rng(seed)
    table = audited_length_table(audit_jsonl)
    lengths = np.arange(LENGTH_MIN, LENGTH_MAX + 1)
    ids, shard, length, success, split = [], [], [], [], []

    def draw(n: int, y: int, is_test: bool, shard_start: int):
        p = table[y] / table[y].sum()
        L = rng.choice(lengths, size=n, p=p)
        # distribute over synthetic shards that obey the split rule
        cand = [s for s in range(shard_start, shard_start + 3000) if bool(is_test_shard(s)) == is_test]
        sh = rng.choice(cand[:400], size=n)
        for k in range(n):
            ids.append(None)  # filled below with a per-shard index
            shard.append(int(sh[k]))
            length.append(int(L[k]))
            success.append(y)
            split.append("test" if is_test else "train")

    draw(train_successes, 1, False, 0)
    draw(train_failures, 0, False, 0)
    draw(test_successes, 1, True, 0)
    draw(test_failures, 0, True, 0)
    shard_arr = np.array(shard, int)
    order = np.lexsort((np.arange(len(shard_arr)), shard_arr))
    counters: dict[int, int] = {}
    id_arr = np.empty(len(shard_arr), dtype=object)
    for k in order:
        s = int(shard_arr[k])
        id_arr[k] = f"{s:05d}/{counters.get(s, 0)}"
        counters[s] = counters.get(s, 0) + 1
    header = {"kind": "SYNTHETIC", "warning": "metadata stand-in for dry runs only; never train or freeze on it",
              "source": str(audit_jsonl), "assumed_counts": {"train_successes": train_successes, "train_failures": train_failures,
                                                               "test_successes": test_successes, "test_failures": test_failures},
              "split_rule": "(shard // 3) % 5 == 0 is test", "eligibility": {"length_min": LENGTH_MIN, "length_max": LENGTH_MAX}, "seed": seed}
    m = Manifest(id_arr, shard_arr, np.array(length, int), np.array(success, int), np.array(split, dtype=object),
                 np.ones(len(shard_arr), bool), header)
    # canonical order: by shard then index
    o = np.lexsort((np.array([int(i.split("/")[1]) for i in m.ids]), m.shard))
    return Manifest(m.ids[o], m.shard[o], m.length[o], m.success[o], m.split[o], m.proprio_ok[o], header)


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    b = sub.add_parser("build", help="build the real manifest from data/kuka/shard_*.npz")
    b.add_argument("--npz-dir", required=True)
    b.add_argument("--out", required=True)
    s = sub.add_parser("synthetic", help="build the SYNTHETIC stand-in from the audit sample")
    s.add_argument("--audit-jsonl", required=True)
    s.add_argument("--out", required=True)
    s.add_argument("--train-failures", type=int, default=6900)
    args = ap.parse_args(argv)
    if args.cmd == "build":
        m = build_manifest(Path(args.npz_dir))
    else:
        m = synthetic_manifest(Path(args.audit_jsonl), train_failures=args.train_failures)
    digest = save_manifest(m, Path(args.out))
    write_json(Path(args.out).with_suffix(".summary.json"), {"sha256": digest, **m.summary(), "header": m.header})
    print(json.dumps({"sha256": digest, **m.summary()}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/design.py
"""Seeded samplers for designs O, A, B and V, and the feasibility certificate.

All samplers work on the metadata manifest only (no features), so the whole selection for all 110
runs is built and certified in seconds, before any training, and can be frozen by hash.

Pairing. For a seed, each outcome pool (train successes, train failures) is permuted once with
``numpy.random.default_rng(seed)``. Every arm of that seed starts from prefixes of these two
permutations, so arms within a seed share episodes as far as their constraints allow, which is the
pairing structure the paired t analysis relies on. Different seeds use different permutations.

Design O  original episode counts (prefixes), as the owner's ``make_arm``.
Design A  exactly T distinct transitions at episode failure fraction phi. The total episode count
          N is a multiple of 20 so phi*N is an integer for every phi in {0, .10, .25, .50, .65}.
          For each candidate N the prefix totals are computed; the N whose total is closest to T is
          repaired to exactly T by same-outcome swaps (a selected episode is exchanged for an
          unselected one whose transition count moves the total toward T; each swap strictly
          reduces the gap, so the loop terminates). Swaps never change phi.
Design B  for lengths l = 3..24, 20*k_l episodes: 18 S + 2 F at phi = .10 and 7 S + 13 F at
          phi = .65. Per-length caps k_l <= min(S_l // 18, F_l // 13) make both arms feasible. The
          vector k is the integer least-squares solution closest to the pooled availability shape
          with sum_l k_l (l - 1) = T / 20 (dynamic programme, exact). Then, within each arm, the
          normalized outcome-conditional length distributions are identical by construction
          (k_l / sum k for both outcomes), and the two arms have the same total histogram 20*k_l.
          If T = 31,600 is infeasible, the budget falls back by the registered rule and Design A
          comparators at that budget are added for both B fractions.
Design V  shared base = first 1,500 successes of the seed; add distinct transitions of further
          successes (V+S) or of failures (V+F) until the arm holds exactly 31,600 transitions (Design
          A's budget), each by prefix + repair. V arms are therefore transition-matched to A.

The certificate (``certify``) re-derives every constraint from the manifest instead of trusting the
sampler's own bookkeeping.
"""

from __future__ import annotations

import argparse
import json
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np

from . import load_config, sha256_json, write_json
from .manifest import LENGTH_MAX, LENGTH_MIN, Manifest, load_manifest

LENGTHS = list(range(LENGTH_MIN, LENGTH_MAX + 1))


class Infeasible(Exception):
    """A registered constraint cannot be met with the available pool."""


@dataclass
class Run:
    run_id: str
    design: str
    arm: str
    phi: float | None
    seed: int
    target_transitions: int | None
    n_success: int
    n_failure: int
    n_episodes: int
    transitions: int
    hist_success: list[int]  # counts over LENGTHS
    hist_failure: list[int]
    ids_sha256: str
    swaps: int = 0
    notes: dict = field(default_factory=dict)
    ids: list[str] = field(default_factory=list)


# ---------------------------------------------------------------- pools and pairing

def train_pools(m: Manifest) -> tuple[np.ndarray, np.ndarray]:
    """Manifest row indices of eligible train successes and failures, in canonical order."""

    tr = m.train_mask()
    return np.where(tr & (m.success == 1))[0], np.where(tr & (m.success == 0))[0]


def seeded_permutations(S: np.ndarray, F: np.ndarray, seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    return S[rng.permutation(len(S))], F[rng.permutation(len(F))]


def _hist(m: Manifest, rows: list[int] | np.ndarray, y: int) -> list[int]:
    rows = np.asarray(rows, int)
    sel = rows[m.success[rows] == y]
    c = Counter(m.length[sel].tolist())
    return [int(c.get(l, 0)) for l in LENGTHS]


def _make_run(m: Manifest, design: str, arm: str, phi: float | None, seed: int, rows: list[int], target: int | None,
              swaps: int = 0, notes: dict | None = None) -> Run:
    rows_arr = np.asarray(rows, int)
    ids = sorted(m.ids[rows_arr].tolist())
    n_s = int(m.success[rows_arr].sum())
    return Run(
        run_id=f"{design}_{arm}_s{seed}", design=design, arm=arm, phi=phi, seed=seed, target_transitions=target,
        n_success=n_s, n_failure=int(len(rows_arr) - n_s), n_episodes=int(len(rows_arr)),
        transitions=int(m.transitions[rows_arr].sum()), hist_success=_hist(m, rows_arr, 1), hist_failure=_hist(m, rows_arr, 0),
        ids_sha256=sha256_json(ids), swaps=swaps, notes=notes or {}, ids=ids,
    )


# ---------------------------------------------------------------- repair

def _best_swap(classes: list[tuple[list[int], list[int]]], trans: np.ndarray, delta: int):
    """The single same-class swap that brings ``delta`` closest to zero and strictly reduces |delta|.

    Ties are broken deterministically: smallest residual, then class order, then smaller incoming
    value, then smaller outgoing value. Returns (class, pos_out, pos_in, change) or None."""

    best = None
    for k, (sel, unsel) in enumerate(classes):
        if not sel or not unsel:
            continue
        vs, vu = np.unique(trans[sel]), np.unique(trans[unsel])
        change = vu[:, None] - vs[None, :]
        resid = np.abs(delta - change)
        ok = resid < abs(delta)
        if not ok.any():
            continue
        i, j = np.unravel_index(int(np.argmin(np.where(ok, resid, np.inf))), resid.shape)
        cand = (int(resid[i, j]), k, int(vu[i]), int(vs[j]))
        if best is None or cand < best:
            best = cand
    if best is None:
        return None
    _, k, v_in, v_out = best
    sel, unsel = classes[k]
    pos_out = max(p for p, e in enumerate(sel) if trans[e] == v_out)      # most recently added
    pos_in = min(p for p, e in enumerate(unsel) if trans[e] == v_in)      # next in permutation order
    return k, pos_out, pos_in, v_in - v_out


def repair_to_target(classes: list[tuple[list[int], list[int]]], trans: np.ndarray, target: int,
                     max_swaps: int = 500) -> tuple[list[tuple[list[int], list[int]]], int]:
    """Swap within classes until the selected transitions sum to ``target`` exactly."""

    classes = [(list(s), list(u)) for s, u in classes]
    total = int(sum(int(trans[s].sum()) for s, _ in classes if s))
    swaps = 0
    while total != target:
        if swaps >= max_swaps:
            raise Infeasible(f"repair exceeded {max_swaps} swaps (gap {target - total})")
        sw = _best_swap(classes, trans, target - total)
        if sw is None:
            raise Infeasible(f"no swap reduces the gap of {target - total} transitions")
        k, pos_out, pos_in, change = sw
        sel, unsel = classes[k]
        e_out, e_in = sel.pop(pos_out), unsel.pop(pos_in)
        sel.append(e_in)
        unsel.append(e_out)
        total += change
        swaps += 1
    return classes, swaps


def _prefix_closest(order: np.ndarray, trans: np.ndarray, target: int) -> int:
    """Prefix length whose transition total is closest to ``target`` (ties: shorter)."""

    c = np.concatenate([[0], np.cumsum(trans[order])])
    return int(np.argmin(np.abs(c - target)))


# ---------------------------------------------------------------- designs

def design_O(m: Manifest, perm_S, perm_F, seed: int, arms: dict) -> list[Run]:
    runs = []
    for arm, c in arms.items():
        n_s, n_f = int(c["n_success"]), int(c["n_failure"])
        if n_s > len(perm_S) or n_f > len(perm_F):
            raise Infeasible(f"O {arm}: need {n_s} S / {n_f} F, have {len(perm_S)} / {len(perm_F)}")
        rows = np.concatenate([perm_S[:n_s], perm_F[:n_f]])
        runs.append(_make_run(m, "O", arm, n_f / (n_s + n_f), seed, rows, None))
    return runs


def design_A_arm(m: Manifest, perm_S, perm_F, seed: int, phi: float, T: int, multiple: int,
                 max_candidates: int, design: str = "A") -> Run:
    trans = m.transitions
    cS = np.concatenate([[0], np.cumsum(trans[perm_S])])
    cF = np.concatenate([[0], np.cumsum(trans[perm_F])])
    cands = []
    for N in range(multiple, len(perm_S) + len(perm_F) + 1, multiple):
        n_f_real = phi * N
        if abs(n_f_real - round(n_f_real)) > 1e-9:
            continue
        n_f = int(round(n_f_real))
        n_s = N - n_f
        if n_s > len(perm_S) or n_f > len(perm_F):
            break
        total = int(cS[n_s] + cF[n_f])
        cands.append((abs(T - total), N, n_s, n_f, total))
    if not cands:
        raise Infeasible(f"A phi={phi}: no episode total is a multiple of {multiple} within the pool")
    cands.sort()
    errors = []
    for gap, N, n_s, n_f, total in cands[:max_candidates]:
        classes = [(list(perm_S[:n_s]), list(perm_S[n_s:])), (list(perm_F[:n_f]), list(perm_F[n_f:]))]
        try:
            classes, swaps = repair_to_target(classes, trans, T)
        except Infeasible as e:
            errors.append(f"N={N}: {e}")
            continue
        rows = classes[0][0] + classes[1][0]
        return _make_run(m, design, f"phi{phi:.2f}", phi, seed, rows, T, swaps,
                         {"episode_total_candidates_tried": len(errors) + 1, "prefix_gap_before_repair": int(gap)})
    raise Infeasible(f"A phi={phi}: all candidates failed: {errors}")


def design_B_shape(avail_S: np.ndarray, avail_F: np.ndarray, phis: list[float], unit: int, T: int) -> dict:
    """Common per-length unit counts k_l for Design B, or the infeasibility report.

    Returns a dict with ``feasible``, ``k`` (list over LENGTHS), ``caps``, ``max_feasible_transitions``,
    and the real-valued target ``k_star`` used by the least-squares DP."""

    need_S = max(int(round((1 - p) * unit)) for p in phis)
    need_F = max(int(round(p * unit)) for p in phis)
    for p in phis:  # the unit must split exactly
        if abs(p * unit - round(p * unit)) > 1e-9:
            raise ValueError(f"unit {unit} does not split phi={p} into integers")
    caps = np.minimum(avail_S // need_S, avail_F // need_F).astype(int)
    w = np.array([l - 1 for l in LENGTHS], int)  # transitions per episode at each length
    max_T = int((caps * w).sum() * unit)
    out = {"unit": unit, "need_S_per_unit": need_S, "need_F_per_unit": need_F, "caps": caps.tolist(),
           "max_feasible_transitions": max_T, "requested_transitions": T}
    if T % unit:
        raise ValueError(f"T={T} is not a multiple of unit {unit}")
    R = T // unit
    if max_T < T:
        out.update(feasible=False, k=None, k_star=None)
        return out
    # water-filling target proportional to pooled availability under caps
    a = (avail_S + avail_F).astype(float)
    lo, hi = 0.0, 1.0
    while (np.minimum(caps, hi * a) * w).sum() < R:
        hi *= 2
    for _ in range(200):
        mid = (lo + hi) / 2
        if (np.minimum(caps, mid * a) * w).sum() < R:
            lo = mid
        else:
            hi = mid
    k_star = np.minimum(caps, hi * a)
    # exact integer least squares: min sum (k_l - k*_l)^2  s.t.  sum k_l w_l = R, 0 <= k_l <= cap_l
    INF = np.inf
    cost = np.full(R + 1, INF)
    cost[0] = 0.0
    choice = np.zeros((len(LENGTHS), R + 1), int)
    for li, (cap, wl, ks) in enumerate(zip(caps, w, k_star)):
        new = np.full(R + 1, INF)
        for k in range(int(cap) + 1):
            shift = k * int(wl)
            if shift > R:
                break
            cand = cost[: R + 1 - shift] + (k - ks) ** 2
            better = cand < new[shift:]
            new[shift:][better] = cand[better]
            choice[li, shift:][better] = k
        cost = new
    if not np.isfinite(cost[R]):
        out.update(feasible=False, k=None, k_star=k_star.tolist(), reason="no integer combination hits the budget exactly")
        return out
    k = np.zeros(len(LENGTHS), int)
    s = R
    for li in range(len(LENGTHS) - 1, -1, -1):
        k[li] = choice[li, s]
        s -= int(k[li] * w[li])
    assert s == 0 and int((k * w).sum()) == R and np.all(k <= caps)
    out.update(feasible=True, k=k.tolist(), k_star=k_star.tolist(), episodes_per_arm=int(k.sum() * unit),
               lengths_used=[l for l, kk in zip(LENGTHS, k) if kk > 0], objective=float(((k - k_star) ** 2).sum()))
    return out


def design_B_arm(m: Manifest, perm_S, perm_F, seed: int, phi: float, k: list[int], unit: int, T: int) -> Run:
    n_f_unit = int(round(phi * unit))
    n_s_unit = unit - n_f_unit
    rows: list[int] = []
    for l, kk in zip(LENGTHS, k):
        if kk == 0:
            continue
        s_l = perm_S[m.length[perm_S] == l][: n_s_unit * kk]
        f_l = perm_F[m.length[perm_F] == l][: n_f_unit * kk]
        if len(s_l) < n_s_unit * kk or len(f_l) < n_f_unit * kk:
            raise Infeasible(f"B phi={phi} length {l}: need {n_s_unit * kk} S / {n_f_unit * kk} F")
        rows += s_l.tolist() + f_l.tolist()
    return _make_run(m, "B", f"phi{phi:.2f}", phi, seed, rows, T, 0, {"unit_counts_k": list(map(int, k))})


def design_V(m: Manifest, perm_S, perm_F, seed: int, base_successes: int, total: int) -> list[Run]:
    """Shared base of successes; add further successes or failures until the arm holds exactly ``total`` transitions."""

    trans = m.transitions
    if base_successes > len(perm_S):
        raise Infeasible(f"V: base needs {base_successes} successes, have {len(perm_S)}")
    base = perm_S[:base_successes]
    base_transitions = int(trans[base].sum())
    added = total - base_transitions
    if added <= 0:
        raise Infeasible(f"V: base already holds {base_transitions} transitions >= total {total}")
    rest_S = perm_S[base_successes:]
    runs = []
    for arm, pool in (("V+S", rest_S), ("V+F", perm_F)):
        n0 = _prefix_closest(pool, trans, added)
        classes, swaps = repair_to_target([(list(pool[:n0]), list(pool[n0:]))], trans, added)
        add_rows = classes[0][0]
        run = _make_run(m, "V", arm, None, seed, np.concatenate([base, add_rows]), total, swaps,
                        {"base_ids_sha256": sha256_json(sorted(m.ids[base].tolist())), "base_episodes": int(len(base)),
                         "base_transitions": base_transitions, "added_episodes": int(len(add_rows)),
                         "added_transitions": int(trans[np.asarray(add_rows, int)].sum())})
        runs.append(run)
    return runs


# ---------------------------------------------------------------- whole selection

def fallback_budget(max_T: int, rule: dict) -> int:
    b = int(np.floor(rule["fraction_of_max"] * max_T / rule["round_to"]) * rule["round_to"])
    return b


def build_selection(m: Manifest, manifest_sha256: str, config: dict) -> dict:
    S, F = train_pools(m)
    avail_S = np.array([int((m.length[S] == l).sum()) for l in LENGTHS])
    avail_F = np.array([int((m.length[F] == l).sum()) for l in LENGTHS])
    dA, dB, dV, dO = config["designs"]["A"], config["designs"]["B"], config["designs"]["V"], config["designs"]["O"]

    shape = design_B_shape(avail_S, avail_F, dB["phis"], dB["unit"], dB["transitions"])
    budget_B = dB["transitions"]
    comparators_at_B = False
    if not shape["feasible"]:
        budget_B = fallback_budget(shape["max_feasible_transitions"], dB["fallback"])
        shape = design_B_shape(avail_S, avail_F, dB["phis"], dB["unit"], budget_B) if budget_B >= dB["unit"] else shape
        comparators_at_B = shape["feasible"]
    runs: list[Run] = []
    problems: list[str] = []
    for seed in config["seeds"]:
        perm_S, perm_F = seeded_permutations(S, F, seed)
        try:
            runs += design_O(m, perm_S, perm_F, seed, dO["arms"])
        except Infeasible as e:
            problems.append(str(e))
        for phi in dA["phis"]:
            try:
                runs.append(design_A_arm(m, perm_S, perm_F, seed, phi, dA["transitions"], dA["episode_multiple"], dA["max_candidate_totals"]))
            except Infeasible as e:
                problems.append(f"seed {seed}: {e}")
        if shape["feasible"]:
            for phi in dB["phis"]:
                try:
                    runs.append(design_B_arm(m, perm_S, perm_F, seed, phi, shape["k"], dB["unit"], budget_B))
                except Infeasible as e:
                    problems.append(f"seed {seed}: {e}")
            if comparators_at_B:
                for phi in dB["phis"]:
                    try:
                        runs.append(design_A_arm(m, perm_S, perm_F, seed, phi, budget_B, dA["episode_multiple"], dA["max_candidate_totals"], design="AB"))
                    except Infeasible as e:
                        problems.append(f"seed {seed}: {e}")
        else:
            problems.append(f"seed {seed}: Design B infeasible at any budget >= {dB['unit']} (max feasible {shape['max_feasible_transitions']})")
        try:
            runs += design_V(m, perm_S, perm_F, seed, dV["base_successes"], dV["total_transitions"])
        except Infeasible as e:
            problems.append(f"seed {seed}: {e}")

    selection = {
        "config_name": config["name"], "config_version": config["version"], "config_sha256": sha256_json(config),
        "manifest_sha256": manifest_sha256, "manifest_kind": m.header.get("kind"),
        "pool": {"train_successes": int(len(S)), "train_failures": int(len(F)), "lengths": LENGTHS,
                 "avail_success_by_length": avail_S.tolist(), "avail_failure_by_length": avail_F.tolist(),
                 "test_episodes": int(m.test_mask().sum())},
        "design_B": {"budget_transitions": budget_B, "reduced_from": dB["transitions"] if budget_B != dB["transitions"] else None,
                     "A_comparators_at_budget": comparators_at_B, **shape},
        "runs": [asdict(r) for r in runs],
        "n_runs": len(runs),
        "problems": problems,
    }
    selection["certificate"] = certify(m, selection, config)
    selection["selection_sha256_without_certificate"] = sha256_json({k: v for k, v in selection.items() if k != "certificate"})
    return selection


# ---------------------------------------------------------------- certificate (independent re-derivation)

def certify(m: Manifest, selection: dict, config: dict) -> dict:
    """Re-check every registered constraint from the manifest; never trust the sampler's fields."""

    idx = m.index_of()
    train = m.train_mask()
    checks: list[dict] = []
    by_seed: dict[int, dict[str, dict]] = {}

    def ok(run_id: str, name: str, passed: bool, detail: str = "") -> None:
        checks.append({"run_id": run_id, "check": name, "pass": bool(passed), "detail": detail})

    for r in selection["runs"]:
        rows = np.array([idx[i] for i in r["ids"]], int)
        ok(r["run_id"], "ids_distinct", len(set(r["ids"])) == len(r["ids"]))
        ok(r["run_id"], "ids_in_eligible_train_split", bool(train[rows].all()))
        ok(r["run_id"], "ids_hash_matches", sha256_json(sorted(r["ids"])) == r["ids_sha256"])
        n_s = int(m.success[rows].sum())
        n_f = len(rows) - n_s
        trans = int(m.transitions[rows].sum())
        ok(r["run_id"], "counts_match_record", n_s == r["n_success"] and n_f == r["n_failure"] and trans == r["transitions"])
        if r["design"] in ("A", "B", "AB"):
            ok(r["run_id"], "transitions_exact", trans == r["target_transitions"], f"{trans} vs {r['target_transitions']}")
            ok(r["run_id"], "phi_exact", abs(n_f - r["phi"] * len(rows)) < 1e-9, f"{n_f}/{len(rows)}")
        if r["design"] == "B":
            hs = np.array([int(((m.length[rows] == l) & (m.success[rows] == 1)).sum()) for l in LENGTHS])
            hf = np.array([int(((m.length[rows] == l) & (m.success[rows] == 0)).sum()) for l in LENGTHS])
            ok(r["run_id"], "B_conditional_length_distributions_equal", bool(np.all(hs * n_f == hf * n_s)),
               "hist_S * n_F == hist_F * n_S at every length")
            by_seed.setdefault(r["seed"], {})[r["arm"]] = {"total_hist": (hs + hf).tolist()}
        if r["design"] == "V":
            ok(r["run_id"], "V_total_transitions_exact", trans == r["target_transitions"], f"{trans} vs {r['target_transitions']}")
        if r["design"] == "O":
            arms = config["designs"]["O"]["arms"][r["arm"]]
            ok(r["run_id"], "O_counts", n_s == arms["n_success"] and n_f == arms["n_failure"])
    for seed, arms in by_seed.items():
        hists = [a["total_hist"] for a in arms.values()]
        ok(f"B_seed{seed}", "B_total_histograms_identical_across_arms", all(h == hists[0] for h in hists))
    # V: shared base across the two V arms of a seed
    v_runs = [r for r in selection["runs"] if r["design"] == "V"]
    for seed in sorted({r["seed"] for r in v_runs}):
        pair = [r for r in v_runs if r["seed"] == seed]
        if len(pair) == 2:
            ok(f"V_seed{seed}", "V_base_shared", pair[0]["notes"]["base_ids_sha256"] == pair[1]["notes"]["base_ids_sha256"])
            a, b = set(pair[0]["ids"]), set(pair[1]["ids"])
            ok(f"V_seed{seed}", "V_additions_disjoint", len(a & b) == pair[0]["notes"]["base_episodes"])
    expected = len(config["seeds"]) * (len(config["designs"]["O"]["arms"]) + len(config["designs"]["A"]["phis"]) + len(config["designs"]["B"]["phis"]) + 2)
    if selection["design_B"].get("A_comparators_at_budget"):
        expected += len(config["seeds"]) * len(config["designs"]["B"]["phis"])
    ok("ALL", "run_count", selection["n_runs"] == expected and not selection["problems"], f"{selection['n_runs']} runs, expected {expected}; problems: {selection['problems']}")
    seeds_seen = sorted({r["seed"] for r in selection["runs"]})
    ok("ALL", "seeds_registered", seeds_seen == sorted(config["seeds"]), str(seeds_seen))
    ok("ALL", "historical_seeds_excluded", not (set(seeds_seen) & set(config["historical_seeds_excluded"])))
    failed = [c for c in checks if not c["pass"]]
    return {"feasible": not failed, "n_checks": len(checks), "n_failed": len(failed), "failed": failed,
            "design_B_budget": selection["design_B"]["budget_transitions"],
            "design_B_reduced": selection["design_B"]["reduced_from"] is not None,
            "checks": checks}


# ---------------------------------------------------------------- reporting

def selection_table(selection: dict) -> str:
    rows = ["| Design | Arm | Seeds | Episodes (min–max) | Successes | Failures | Transitions | Swaps (max) |", "|---|---|---|---|---|---|---|---|"]
    groups: dict[tuple[str, str], list[dict]] = {}
    for r in selection["runs"]:
        groups.setdefault((r["design"], r["arm"]), []).append(r)
    for (d, a), rs in groups.items():
        ne = [r["n_episodes"] for r in rs]
        ns = [r["n_success"] for r in rs]
        nf = [r["n_failure"] for r in rs]
        tr = sorted({r["transitions"] for r in rs})
        trs = f"{tr[0]:,}" if len(tr) == 1 else f"{tr[0]:,}–{tr[-1]:,}"
        rows.append(f"| {d} | {a} | {len(rs)} | {min(ne):,}–{max(ne):,} | {min(ns):,}–{max(ns):,} | {min(nf):,}–{max(nf):,} | {trs} | {max(r['swaps'] for r in rs)} |")
    return "\n".join(rows)


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Build and certify the registered selection for all designs and seeds.")
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--config", default=None)
    ap.add_argument("--out", required=True, help="selection JSON (ids inline)")
    ap.add_argument("--allow-synthetic", action="store_true", help="permit a SYNTHETIC manifest (dry runs only)")
    args = ap.parse_args(argv)
    m, digest = load_manifest(Path(args.manifest))
    if m.header.get("kind") == "SYNTHETIC" and not args.allow_synthetic:
        raise SystemExit("refusing a SYNTHETIC manifest without --allow-synthetic")
    config = load_config(args.config)
    sel = build_selection(m, digest, config)
    write_json(Path(args.out), sel)
    cert = sel["certificate"]
    summary = {k: v for k, v in cert.items() if k != "checks"}
    summary["design_B"] = {k: v for k, v in sel["design_B"].items() if k not in ("k_star",)}
    write_json(Path(args.out).with_name(Path(args.out).stem + ".certificate.json"), {**summary, "checks": cert["checks"]})
    Path(args.out).with_suffix(".md").write_text(selection_table(sel) + "\n")
    print(json.dumps(summary, indent=1))
    print(selection_table(sel))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/weights.py
"""Frozen evaluation support and overlap weights (Protocol §2, "Length standardization").

Support: exact-length bins l = 3..24. A length is retained when the evaluation set has at least
``min_eval_per_outcome_per_length`` episodes of each outcome and the training pool has both outcomes
at that length. Support is fixed from labels and lengths only, before any model score exists.

Weights: with n_{y,l} retained evaluation episodes of outcome y at length l, n_y = sum_l n_{y,l},
r_{y,l} = n_{y,l} / n_y and the overlap target q_l = min(r_{0,l}, r_{1,l}) / sum_k min(r_{0,k}, r_{1,k}),
episode i gets a_i = pi_{Y_i} q_{L_i} / n_{Y_i, L_i}, where pi_1 = pi is the prevalence among all
eligible evaluation episodes and pi_0 = 1 - pi. Then sum_i a_i = 1, both outcomes share the length
distribution q, and the weighted truth is exactly pi. These are evaluation weights only.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

from . import load_config, sha256_json, write_json
from .design import LENGTHS
from .manifest import Manifest, load_manifest


def evaluation_support(m: Manifest, config: dict) -> dict:
    ev = config["evaluation"]
    te, tr = m.test_mask(), m.train_mask()
    pi = float(m.success[te].mean())
    counts = {y: np.array([int(((m.length == l) & te & (m.success == y)).sum()) for l in LENGTHS]) for y in (0, 1)}
    train_counts = {y: np.array([int(((m.length == l) & tr & (m.success == y)).sum()) for l in LENGTHS]) for y in (0, 1)}
    keep = np.ones(len(LENGTHS), bool)
    reasons: dict[int, list[str]] = {}
    for li, l in enumerate(LENGTHS):
        for y in (0, 1):
            if counts[y][li] < ev["min_eval_per_outcome_per_length"]:
                keep[li] = False
                reasons.setdefault(l, []).append(f"eval outcome {y}: {counts[y][li]} < {ev['min_eval_per_outcome_per_length']}")
            if train_counts[y][li] < ev["min_train_per_outcome_per_length"]:
                keep[li] = False
                reasons.setdefault(l, []).append(f"train outcome {y}: {train_counts[y][li]} < {ev['min_train_per_outcome_per_length']}")
    n_y = {y: int(counts[y][keep].sum()) for y in (0, 1)}
    if min(n_y.values()) == 0:
        raise ValueError("no supported lengths for one outcome")
    r = {y: counts[y] / n_y[y] for y in (0, 1)}
    overlap = np.where(keep, np.minimum(r[0], r[1]), 0.0)
    q = overlap / overlap.sum()
    pi_y = {1: pi, 0: 1 - pi}
    weights: dict[str, float] = {}
    for i in np.where(te)[0]:
        li = int(m.length[i]) - LENGTHS[0]
        if not keep[li]:
            continue
        y = int(m.success[i])
        weights[str(m.ids[i])] = float(pi_y[y] * q[li] / counts[y][li])
    a = np.array(list(weights.values()))
    y_w = np.array([m.success[m.index_of()[i]] for i in weights])
    ess = float(1.0 / (a ** 2).sum())
    ess_by = {y: float(a[y_w == y].sum() ** 2 / (a[y_w == y] ** 2).sum()) for y in (0, 1)}
    out = {
        "prevalence_pi": pi,
        "eligible_test_episodes": int(te.sum()),
        "retained_test_episodes": int(len(weights)),
        "excluded_test_episodes": int(te.sum() - len(weights)),
        "support_lengths": [l for l, k in zip(LENGTHS, keep) if k],
        "excluded_lengths": {str(l): v for l, v in reasons.items()},
        "eval_counts_by_length": {"success": counts[1].tolist(), "failure": counts[0].tolist(), "lengths": LENGTHS},
        "train_counts_by_length": {"success": train_counts[1].tolist(), "failure": train_counts[0].tolist()},
        "n_y_retained": {"success": n_y[1], "failure": n_y[0]},
        "q_by_length": dict(zip(map(str, LENGTHS), q.tolist())),
        "weight_sum": float(a.sum()),
        "weighted_truth": float((a * y_w).sum()),
        "ess": ess,
        "ess_by_outcome": {"success": ess_by[1], "failure": ess_by[0]},
        "weights": weights,
    }
    out["sha256"] = sha256_json({k: v for k, v in out.items() if k != "sha256"})
    assert abs(out["weight_sum"] - 1) < 1e-9 and abs(out["weighted_truth"] - pi) < 1e-9
    return out


def support_table(s: dict) -> str:
    rows = ["| Length | Eval S | Eval F | Train S | Train F | Retained | q_l |", "|---|---|---|---|---|---|---|"]
    for li, l in enumerate(LENGTHS):
        kept = l in s["support_lengths"]
        rows.append(f"| {l} | {s['eval_counts_by_length']['success'][li]} | {s['eval_counts_by_length']['failure'][li]} | "
                    f"{s['train_counts_by_length']['success'][li]} | {s['train_counts_by_length']['failure'][li]} | "
                    f"{'yes' if kept else 'no'} | {s['q_by_length'][str(l)]:.4f} |")
    rows.append(f"\nπ = {s['prevalence_pi']:.6f}; retained {s['retained_test_episodes']} of {s['eligible_test_episodes']} evaluation episodes; "
                f"ESS = {s['ess']:.0f} (successes {s['ess_by_outcome']['success']:.0f}, failures {s['ess_by_outcome']['failure']:.0f}); "
                f"Σa = {s['weight_sum']:.6f}; weighted truth = {s['weighted_truth']:.6f}.")
    return "\n".join(rows)


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Freeze the evaluation support and weights from the manifest.")
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--config", default=None)
    ap.add_argument("--out", required=True)
    args = ap.parse_args(argv)
    m, digest = load_manifest(Path(args.manifest))
    s = evaluation_support(m, load_config(args.config))
    s["manifest_sha256"] = digest
    s["manifest_kind"] = m.header.get("kind")
    write_json(Path(args.out), s)
    Path(args.out).with_suffix(".md").write_text(support_table(s) + "\n")
    print(json.dumps({k: v for k, v in s.items() if k not in ("weights", "q_by_length", "eval_counts_by_length", "train_counts_by_length")}, indent=1))
    print(support_table(s))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/analyze.py
"""The registered decision contract for Experiment 2 (Protocol §2, "Analysis plan").

Inputs are frozen artefacts (selection, support, config, optional freeze manifest) and one JSON per
training run written by ``run_registered``. The analyzer:

1. validates every run against the frozen selection (run id, hashes, evaluation ids, probabilities);
2. computes weighted thresholded success R_ds = sum_i a_i 1[p_ids > 0.5] per run, plus the
   mean-probability and unweighted descriptives, Brier score and error against truth;
3. forms paired seed contrasts D_ds = 100 (R_d,.10,s - R_d,.65,s) for designs O, A, B, the V contrast
   100 (R_V+S,s - R_V+F,s), and the differences O - A and A - B (A at B's budget when comparators exist);
4. applies paired t inference: support if the one-sided 95% lower bound exceeds the margin (3 points),
   absence if the 90% CI lies inside (-3, 3) (two one-sided tests at 5%); the ten secondary p-values
   (support and equivalence for O, B, V, O - A, A - B) are Holm-adjusted at familywise 0.05;
5. makes every decision that depends on a missing or failed run inconclusive and reports worst-case
   bounds with rates in [0, 1] instead of analysing only the completed runs;
6. maps the decision pattern to the registered interpretation table.

Nothing here reads a model or a feature; it is pure arithmetic on frozen inputs, so it can be dry-run
on synthetic scores (``dryrun``) and the operating characteristics attached to the registration.
"""

from __future__ import annotations

import argparse
import json
from itertools import product
from pathlib import Path

import numpy as np
from scipy import stats

from . import load_config, sha256_json, write_json

REQUIRED_RUN_FIELDS = ("run_id", "design", "arm", "seed", "status", "selection_sha256", "config_sha256")
COMPLETED = "completed"


class AnalysisError(Exception):
    """Frozen inputs or run files are inconsistent; no decision is made."""


# ---------------------------------------------------------------- validation

def validate_runs(runs: list[dict], selection: dict, support: dict, config: dict) -> list[str]:
    """Return a list of hard problems (empty if the run set is admissible)."""

    problems: list[str] = []
    expected = {r["run_id"]: r for r in selection["runs"]}
    seen: set[str] = set()
    sel_hash = selection["selection_sha256_without_certificate"]
    cfg_hash = sha256_json(config)
    weight_ids = set(support["weights"])
    for r in runs:
        for f in REQUIRED_RUN_FIELDS:
            if f not in r:
                problems.append(f"{r.get('run_id', '?')}: missing field {f}")
        rid = r.get("run_id")
        if rid not in expected:
            problems.append(f"{rid}: not in the frozen selection")
            continue
        if rid in seen:
            problems.append(f"{rid}: duplicate run file")
        seen.add(rid)
        if r.get("selection_sha256") != sel_hash:
            problems.append(f"{rid}: selection hash {str(r.get('selection_sha256'))[:12]} != frozen {sel_hash[:12]} (stale artefact)")
        if r.get("config_sha256") != cfg_hash:
            problems.append(f"{rid}: config hash does not match the frozen config")
        if r.get("ids_sha256") not in (None, expected[rid]["ids_sha256"]):
            problems.append(f"{rid}: training ids hash differs from the frozen selection")
        if r.get("status") == COMPLETED:
            ids, p = r.get("eval_ids"), r.get("p")
            if not ids or p is None or len(ids) != len(p):
                problems.append(f"{rid}: eval_ids and p missing or misaligned")
                continue
            if len(set(ids)) != len(ids):
                problems.append(f"{rid}: duplicate evaluation ids")
            if not weight_ids <= set(ids):
                problems.append(f"{rid}: {len(weight_ids - set(ids))} supported evaluation episodes have no score")
            arr = np.asarray(p, float)
            if not np.all(np.isfinite(arr)) or arr.min() < 0 or arr.max() > 1:
                problems.append(f"{rid}: probabilities outside [0, 1] or non-finite")
        elif r.get("status") not in ("infrastructure_failure", "algorithmic_failure"):
            problems.append(f"{rid}: unknown status {r.get('status')}")
    return problems


# ---------------------------------------------------------------- per-run rates

def run_rates(r: dict, support: dict, truth: dict[str, int], threshold: float, with_auroc: bool = True) -> dict:
    ids = r["eval_ids"]
    p = np.asarray(r["p"], float)
    w = np.array([support["weights"].get(i, 0.0) for i in ids])
    y = np.array([truth[i] for i in ids], float)
    h = (p > threshold).astype(float)
    pi = support["prevalence_pi"]
    R = float((w * h).sum())
    out = {
        "R_thr": R, "R_meanp": float((w * p).sum()),
        "R_thr_unweighted": float(h.mean()), "R_meanp_unweighted": float(p.mean()),
        "brier_w": float((w * (p - y) ** 2).sum()), "signed_error_vs_truth": R - pi, "abs_error_vs_truth": abs(R - pi),
        "p_on_failures_w": float((w * p)[y == 0].sum() / w[y == 0].sum()), "p_on_successes_w": float((w * p)[y == 1].sum() / w[y == 1].sum()),
    }
    out["auroc_unweighted"] = None
    if with_auroc:
        try:
            from sklearn.metrics import roc_auc_score
            out["auroc_unweighted"] = float(roc_auc_score(y, p))
        except Exception:  # sklearn optional
            pass
    return out


# ---------------------------------------------------------------- inference

def paired_t(d: np.ndarray, margin: float, alpha: float, eq_level: float) -> dict:
    d = np.asarray(d, float)
    n = int(len(d))
    if n < 2:
        return {"n": n, "mean": float(d.mean()) if n else None, "sd": None, "se": None, "lower95_one_sided": None,
                "ci90": None, "p_support": None, "p_equivalence": None}
    mean, sd = float(d.mean()), float(d.std(ddof=1))
    se = sd / np.sqrt(n)
    df = n - 1
    t_a = stats.t.ppf(1 - alpha, df)
    t_eq = stats.t.ppf(1 - (1 - eq_level) / 2, df)
    if se == 0:
        p_sup = 0.0 if mean > margin else 1.0
        p_eq = 0.0 if abs(mean) < margin else 1.0
    else:
        p_sup = float(stats.t.sf((mean - margin) / se, df))                     # H0: delta <= margin
        p1 = float(stats.t.sf((mean + margin) / se, df))                        # H0: delta <= -margin
        p2 = float(stats.t.cdf((mean - margin) / se, df))                       # H0: delta >= margin
        p_eq = max(p1, p2)
    return {"n": n, "mean": mean, "sd": sd, "se": float(se), "lower95_one_sided": float(mean - t_a * se),
            "ci90": [float(mean - t_eq * se), float(mean + t_eq * se)], "p_support": p_sup, "p_equivalence": p_eq}


def holm(pvals: dict[str, float], alpha: float) -> dict[str, dict]:
    items = [(k, v) for k, v in pvals.items() if v is not None]
    m = len(items)
    order = sorted(items, key=lambda kv: kv[1])
    adjusted: dict[str, dict] = {}
    running = 0.0
    for i, (k, p) in enumerate(order):
        adj = min(1.0, (m - i) * p)
        running = max(running, adj)
        adjusted[k] = {"p": p, "p_holm": running, "reject": running < alpha}
    for k, v in pvals.items():
        if v is None:
            adjusted[k] = {"p": None, "p_holm": None, "reject": False}
    return adjusted


# ---------------------------------------------------------------- contracts

def contrast_specs(selection: dict) -> dict[str, list[tuple[str, str, float]]]:
    """Quantity -> list of (design, arm, coefficient) applied to R and multiplied by 100."""

    a_for_b = "AB" if selection["design_B"].get("A_comparators_at_budget") else "A"
    specs = {
        "A": [("A", "phi0.10", 1), ("A", "phi0.65", -1)],
        "O": [("O", "phi0.10", 1), ("O", "phi0.65", -1)],
        "B": [("B", "phi0.10", 1), ("B", "phi0.65", -1)],
        "V": [("V", "V+S", 1), ("V", "V+F", -1)],
        "O_minus_A": [("O", "phi0.10", 1), ("O", "phi0.65", -1), ("A", "phi0.10", -1), ("A", "phi0.65", 1)],
        "A_minus_B": [(a_for_b, "phi0.10", 1), (a_for_b, "phi0.65", -1), ("B", "phi0.10", -1), ("B", "phi0.65", 1)],
    }
    if a_for_b == "AB":
        specs["AB"] = [("AB", "phi0.10", 1), ("AB", "phi0.65", -1)]
    return specs


def seed_contrasts(rates: dict[tuple[str, str, int], dict], spec, seeds, key="R_thr"):
    """Per-seed contrast values (NaN when a run is missing) and the missing (design, arm, seed) list."""

    vals, missing = [], []
    for s in seeds:
        v = 0.0
        ok = True
        for d, a, c in spec:
            r = rates.get((d, a, s))
            if r is None:
                ok = False
                missing.append((d, a, s))
            else:
                v += c * r[key]
        vals.append(100 * v if ok else np.nan)
    return np.array(vals), missing


def worst_case_bounds(rates, spec, seeds, key="R_thr") -> list[float]:
    """Bounds on the seed-mean contrast when missing rates range over [0, 1]."""

    lo = hi = 0.0
    n = len(seeds)
    for s in seeds:
        for d, a, c in spec:
            r = rates.get((d, a, s))
            if r is None:
                lo += min(0.0, 100 * c) / n
                hi += max(0.0, 100 * c) / n
            else:
                lo += 100 * c * r[key] / n
                hi += 100 * c * r[key] / n
    return [float(lo), float(hi)]


def interpret(dec: dict[str, str]) -> list[str]:
    """Registered outcome-to-interpretation table (Protocol §2)."""

    rows = []
    g = dec.get
    if g("O") == "supported" and g("A") == "equivalent" and g("O_minus_A") == "supported":
        rows.append("Effect is sensitive to the transition-budget intervention; changed episode counts also matter.")
    if g("A") == "supported":
        rows.append("Effect survives transition matching and evaluation standardization; training length shortcuts remain possible.")
    if g("A") == "supported" and g("B") == "equivalent" and g("A_minus_B") == "supported":
        rows.append("Training-length balancing attenuates the effect, conditional on matched budgets and reported population changes.")
    if g("B") == "supported":
        rows.append("Effect persists after removing outcome–length association in WM training; this does not isolate dynamics error.")
    if g("A") == "inconclusive" or g("B") == "inconclusive":
        rows.append("Insufficient evidence for the inconclusive design(s); no 'effect erased' claim.")
    if g("V") == "supported":
        rows.append("Failure additions lower estimates relative to success additions; benefit requires error/discrimination evidence.")
    return rows or ["No registered interpretation row applies; report descriptively."]


def analyze(runs: list[dict], selection: dict, support: dict, config: dict, judge: dict | None = None,
            truth: dict[str, int] | None = None, with_auroc: bool = True) -> dict:
    problems = validate_runs(runs, selection, support, config)
    if problems:
        raise AnalysisError("; ".join(problems[:20]) + (f" (+{len(problems) - 20} more)" if len(problems) > 20 else ""))
    dc = config["decision"]
    margin, alpha, eq_level = dc["margin_points"], dc["alpha_support_one_sided"], dc["equivalence_ci_level"]
    seeds = list(config["seeds"])
    if truth is None:
        truth = {}
    expected_ids = {r["run_id"] for r in selection["runs"]}
    present = {r["run_id"]: r for r in runs}
    missing_runs = sorted(expected_ids - set(present))
    failed_runs = {r["run_id"]: r["status"] for r in runs if r["status"] != COMPLETED}

    rates: dict[tuple[str, str, int], dict] = {}
    for r in runs:
        if r["status"] == COMPLETED:
            rates[(r["design"], r["arm"], r["seed"])] = run_rates(r, support, truth, config["evaluation"]["threshold"], with_auroc)

    specs = contrast_specs(selection)
    quantities: dict[str, dict] = {}
    for q, spec in specs.items():
        d, missing = seed_contrasts(rates, spec, seeds)
        d_meanp, _ = seed_contrasts(rates, spec, seeds, key="R_meanp")
        complete = ~np.isnan(d)
        entry = {"spec": spec, "per_seed": [None if np.isnan(x) else float(x) for x in d],
                 "missing_runs": sorted({f"{dd}_{a}_s{s}" for dd, a, s in missing}),
                 "complete": bool(complete.all())}
        if complete.all():
            entry["t"] = paired_t(d, margin, alpha, eq_level)
            entry["t_meanp_sensitivity"] = paired_t(d_meanp, margin, alpha, eq_level)
        else:
            entry["t"] = None
            entry["worst_case_bounds_on_mean"] = worst_case_bounds(rates, spec, seeds)
            entry["descriptive_completed_seeds_only"] = paired_t(d[complete], margin, alpha, eq_level) if complete.sum() >= 2 else None
        quantities[q] = entry

    # primary
    decisions: dict[str, str] = {}
    prim = quantities[dc["primary"]]
    if prim["t"] is None:
        decisions[dc["primary"]] = "inconclusive"
    else:
        t = prim["t"]
        if t["lower95_one_sided"] > margin:
            decisions[dc["primary"]] = "supported"
        elif t["ci90"][0] > -margin and t["ci90"][1] < margin:
            decisions[dc["primary"]] = "equivalent"
        else:
            decisions[dc["primary"]] = "inconclusive"
    # secondary with Holm over ten p-values
    pvals: dict[str, float | None] = {}
    for q in dc["secondary"]:
        t = quantities[q]["t"]
        pvals[f"{q}:support"] = None if t is None else t["p_support"]
        pvals[f"{q}:equivalence"] = None if t is None else t["p_equivalence"]
    adj = holm(pvals, dc["holm_alpha"])
    for q in dc["secondary"]:
        if quantities[q]["t"] is None:
            decisions[q] = "inconclusive"
        elif adj[f"{q}:support"]["reject"]:
            decisions[q] = "supported"
        elif adj[f"{q}:equivalence"]["reject"]:
            decisions[q] = "equivalent"
        else:
            decisions[q] = "inconclusive"
    # descriptive arm table
    arm_table: dict[str, dict] = {}
    for (d, a, s), r in rates.items():
        arm_table.setdefault(f"{d}_{a}", []).append(r)
    arm_summary = {k: {m: {"mean": float(np.mean([x[m] for x in v if x[m] is not None])), "sd": float(np.std([x[m] for x in v if x[m] is not None], ddof=1)) if len(v) > 1 else None}
                       for m in ("R_thr", "R_meanp", "R_thr_unweighted", "R_meanp_unweighted", "brier_w", "signed_error_vs_truth", "abs_error_vs_truth", "auroc_unweighted") if any(x[m] is not None for x in v)}
                   for k, v in arm_table.items()}
    judge_ref = None
    if judge is not None and truth:
        judge_ref = run_rates({"eval_ids": judge["eval_ids"], "p": judge["p_real"]}, support, truth, config["evaluation"]["threshold"])
        y = np.array([truth[i] for i in judge["eval_ids"]])
        h = np.asarray(judge["p_real"]) > config["evaluation"]["threshold"]
        judge_ref.update({"false_positive_rate": float(h[y == 0].mean()), "false_negative_rate": float((~h[y == 1]).mean()),
                          "validity_gate_auroc_min": config["judge"]["validity_auroc"],
                          "validity_gate_passed": bool(judge_ref["auroc_unweighted"] is not None and judge_ref["auroc_unweighted"] >= config["judge"]["validity_auroc"])})
    return {
        "config_sha256": sha256_json(config), "selection_sha256": selection["selection_sha256_without_certificate"], "support_sha256": support["sha256"],
        "margin_points": margin, "seeds": seeds, "n_runs_expected": len(expected_ids), "n_runs_completed": len(rates),
        "missing_runs": missing_runs, "failed_runs": failed_runs,
        "decisions": decisions, "interpretation": interpret(decisions),
        "holm": adj, "quantities": quantities, "arm_summary": arm_summary, "judge_on_real": judge_ref,
        "prevalence_pi": support["prevalence_pi"], "ess": support["ess"], "retained_eval_episodes": support["retained_test_episodes"],
    }


# ---------------------------------------------------------------- report

def report_markdown(res: dict) -> str:
    L = [f"# Experiment 2 decision report", "",
         f"Runs completed {res['n_runs_completed']} / {res['n_runs_expected']}; missing {len(res['missing_runs'])}; failed {len(res['failed_runs'])}. "
         f"Margin ±{res['margin_points']:g} points. π = {res['prevalence_pi']:.4f}; ESS = {res['ess']:.0f}; retained evaluation episodes = {res['retained_eval_episodes']}.", "",
         "## Decisions", "", "| Quantity | Role | Mean (points) | SD | One-sided 95% lower | 90% CI | p support | p equiv | Holm p (support / equiv) | Decision |", "|---|---|---|---|---|---|---|---|---|---|"]
    for q, e in res["quantities"].items():
        role = "primary" if q == "A" else ("secondary" if q in res["decisions"] else "descriptive")
        dec = res["decisions"].get(q, "—")
        if e["t"] is None:
            wc = e.get("worst_case_bounds_on_mean")
            L.append(f"| {q} | {role} | — | — | — | — | — | — | — | **{dec}** (missing: {', '.join(e['missing_runs'])}; worst-case mean ∈ [{wc[0]:.1f}, {wc[1]:.1f}]) |")
            continue
        t = e["t"]
        hs = res["holm"].get(f"{q}:support", {}).get("p_holm")
        he = res["holm"].get(f"{q}:equivalence", {}).get("p_holm")
        holm_txt = "n/a" if hs is None else f"{hs:.4f} / {he:.4f}"
        L.append(f"| {q} | {role} | {t['mean']:.2f} | {t['sd']:.2f} | {t['lower95_one_sided']:.2f} | [{t['ci90'][0]:.2f}, {t['ci90'][1]:.2f}] | {t['p_support']:.4f} | {t['p_equivalence']:.4f} | {holm_txt} | **{dec}** |")
    L += ["", "## Interpretation (registered table)", ""] + [f"- {s}" for s in res["interpretation"]]
    L += ["", "## Arm summary (weighted thresholded success, mean over seeds)", "", "| Arm | R_thr | R_meanp | unweighted R_thr | Brier (w) | signed error vs truth | AUROC |", "|---|---|---|---|---|---|---|"]
    for k, v in sorted(res["arm_summary"].items()):
        au = v.get("auroc_unweighted", {}).get("mean")
        L.append(f"| {k} | {v['R_thr']['mean']:.4f} | {v['R_meanp']['mean']:.4f} | {v['R_thr_unweighted']['mean']:.4f} | {v['brier_w']['mean']:.4f} | {v['signed_error_vs_truth']['mean']:+.4f} | {'' if au is None else f'{au:.3f}'} |")
    if res["judge_on_real"]:
        j = res["judge_on_real"]
        L += ["", "## Judge on real held-out trajectories", "",
              f"Weighted thresholded rate {j['R_thr']:.4f} (truth {res['prevalence_pi']:.4f}); Brier {j['brier_w']:.4f}; AUROC {j['auroc_unweighted']:.3f}; "
              f"FPR {j['false_positive_rate']:.3f}; FNR {j['false_negative_rate']:.3f}; validity gate (AUROC ≥ {j['validity_gate_auroc_min']}) {'passed' if j['validity_gate_passed'] else 'FAILED'}."]
    if res["failed_runs"]:
        L += ["", "## Failed runs", ""] + [f"- {k}: {v}" for k, v in res["failed_runs"].items()]
    return "\n".join(L) + "\n"


def load_runs(run_dir: Path) -> list[dict]:
    return [json.loads(p.read_text()) for p in sorted(Path(run_dir).glob("*_s*.json"))]


def truth_from_manifest(manifest_path: Path) -> dict[str, int]:
    from .manifest import load_manifest
    m, _ = load_manifest(manifest_path)
    return {str(i): int(y) for i, y in zip(m.ids, m.success)}


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Run the registered Experiment 2 decision contract.")
    ap.add_argument("--runs", required=True, help="directory of per-run JSON files")
    ap.add_argument("--selection", required=True)
    ap.add_argument("--support", required=True)
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--config", default=None)
    ap.add_argument("--judge", default=None, help="judge.json with eval_ids and p_real")
    ap.add_argument("--freeze", default=None, help="freeze manifest; analysis refuses to run if hashes differ or sign-offs are missing")
    ap.add_argument("--out", required=True)
    args = ap.parse_args(argv)
    config = load_config(args.config)
    selection = json.loads(Path(args.selection).read_text())
    support = json.loads(Path(args.support).read_text())
    if args.freeze:
        from .freeze import verify_freeze
        verify_freeze(json.loads(Path(args.freeze).read_text()), config=config, selection=selection, support=support, require_signoff=True)
    judge = json.loads(Path(args.judge).read_text()) if args.judge else None
    res = analyze(load_runs(Path(args.runs)), selection, support, config, judge, truth_from_manifest(Path(args.manifest)))
    write_json(Path(args.out), res)
    Path(args.out).with_suffix(".md").write_text(report_markdown(res))
    print(report_markdown(res))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/dryrun.py
"""Synthetic dry run and operating characteristics of the frozen Experiment 2 analyzer.

Per-episode judge scores are simulated for every run of the certified selection under declared true
contrasts, then passed through the same validation and analyzer (``analyze.analyze``) that will score
the real study. Nothing in the analyzer is bypassed, so the registration documents exactly what the
frozen code decides under each truth.

Generative model (rate scale). Episode i in run (d, s) is a predicted success with probability
    pi_i(d, s) = base_{Y_i} + offset_d + u_s + v_{d,s},
base_1 = 0.62, base_0 = 0.45 (weighted base rate 0.51, as the owner's zero-failure arm), offset_d
encodes the declared contrasts, u_s ~ N(0, 0.01^2) is a shared seed effect, and v_{d,s} is arm-by-seed
noise whose SD is calibrated so that the paired seed contrast has the requested SD (1.88 historical,
3 sensitivity, 5 stress) after the binomial component implied by the frozen weights. The judge
probability is drawn so that 1[p > 0.5] equals the simulated outcome and the mean probability carries
half the contrast, which keeps the mean-probability sensitivity distinct from the primary response.
"""

from __future__ import annotations

import argparse
import json
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np

from . import PACKAGE_ROOT, load_config, sha256_json, write_json
from .analyze import analyze
from .manifest import load_manifest

BASE = {1: 0.62, 0: 0.45}
SEED_SD = 0.01

# true contrasts in percentage points: Δ_O, Δ_A, Δ_B, Δ_V; optional missing run and SD override
SCENARIOS: dict[str, dict] = {
    "global_null": dict(O=0, A=0, B=0, V=0),
    "A_at_margin_3": dict(O=0, A=3, B=0, V=0),
    "A_at_minus_margin_-3": dict(O=0, A=-3, B=0, V=0),
    "A_5_planning": dict(O=10, A=5, B=5, V=5),
    "historical_10": dict(O=10, A=10, B=10, V=10),
    "transition_confound": dict(O=10, A=0, B=0, V=0),
    "length_shortcut": dict(O=10, A=10, B=0, V=10),
    "persists_after_balancing": dict(O=8, A=8, B=8, V=8),
    "reverse_-5": dict(O=0, A=-5, B=0, V=0),
    "missing_run_historical_10": dict(O=10, A=10, B=10, V=10, missing=["A_phi0.65_s104"]),
    "A_5_stress_sd5": dict(O=10, A=5, B=5, V=5, sd=5.0),
}
INTERPRETATION_KEYS = {
    "transition_budget_sensitive": "Effect is sensitive to the transition-budget intervention",
    "survives_matching": "Effect survives transition matching",
    "length_balancing_attenuates": "Training-length balancing attenuates",
    "persists_after_balancing": "Effect persists after removing outcome",
    "insufficient_evidence": "Insufficient evidence",
    "V_failures_lower": "Failure additions lower estimates",
}


def arm_offset(design: str, arm: str, truth: dict, selection: dict) -> float:
    """Rate offset (fraction) for an arm so that the .10 - .65 contrast equals the declared points."""

    if design == "V":
        return 0.0 if arm == "V+S" else -truth["V"] / 100
    key = "A" if design == "AB" else design
    phi = float(arm.replace("phi", ""))
    slope = truth[key] / 100 / 0.55
    return -slope * (phi - 0.10)


def calibrated_arm_seed_sd(target_points: float, support: dict) -> float:
    a = np.array(list(support["weights"].values()))
    binom_var_points = 2 * (100 ** 2) * 0.25 * float((a ** 2).sum())  # two arms, p(1-p) <= 0.25
    rest = max(target_points ** 2 - binom_var_points, 0.0)
    return float(np.sqrt(rest / 2) / 100)


def synthetic_runs(rng: np.random.Generator, selection: dict, support: dict, config: dict, truth: dict,
                   eval_ids: list[str], y_eval: np.ndarray, sd_points: float) -> list[dict]:
    sel_hash = selection["selection_sha256_without_certificate"]
    cfg_hash = sha256_json(config)
    sd_v = calibrated_arm_seed_sd(sd_points, support)
    u = {s: rng.normal(0, SEED_SD) for s in config["seeds"]}
    base = np.where(y_eval == 1, BASE[1], BASE[0])
    missing = set(truth.get("missing", []))
    runs = []
    for r in selection["runs"]:
        rec = {"run_id": r["run_id"], "design": r["design"], "arm": r["arm"], "seed": r["seed"], "selection_sha256": sel_hash,
               "config_sha256": cfg_hash, "ids_sha256": r["ids_sha256"], "status": "completed"}
        if r["run_id"] in missing:
            rec["status"] = "infrastructure_failure"
            runs.append(rec)
            continue
        pi = np.clip(base + arm_offset(r["design"], r["arm"], truth, selection) + u[r["seed"]] + rng.normal(0, sd_v), 0.01, 0.99)
        h = rng.random(len(pi)) < pi
        unif = rng.random(len(pi))
        p = np.where(h, 0.5 + 0.5 * unif, 0.5 * unif)
        rec.update(eval_ids=eval_ids, p=p.tolist())
        runs.append(rec)
    return runs


_CACHE: dict = {}


def _frozen(paths: dict) -> tuple:
    key = tuple(sorted(paths.items()))
    if key not in _CACHE:
        config = load_config(paths["config"])
        selection = json.loads(Path(paths["selection"]).read_text())
        support = json.loads(Path(paths["support"]).read_text())
        m, _ = load_manifest(Path(paths["manifest"]))
        te = m.test_mask()
        _CACHE[key] = (config, selection, support, m.ids[te].tolist(), m.success[te], {str(i): int(y) for i, y in zip(m.ids, m.success)})
    return _CACHE[key]


def _stable_int(name: str) -> int:
    import hashlib
    return int(hashlib.sha256(name.encode()).hexdigest()[:8], 16)


def _one(args) -> dict:
    name, truth, sd_points, dataset_seed, paths = args
    config, selection, support, eval_ids, y_eval, truth_map = _frozen(paths)
    rng = np.random.default_rng([dataset_seed, _stable_int(name)])
    runs = synthetic_runs(rng, selection, support, config, truth, eval_ids, y_eval, truth.get("sd", sd_points))
    res = analyze(runs, selection, support, config, truth=truth_map, with_auroc=False)
    return {"scenario": name, "dataset": dataset_seed, "decisions": res["decisions"], "interpretation": res["interpretation"],
            "A_mean": res["quantities"]["A"]["t"]["mean"] if res["quantities"]["A"]["t"] else None,
            "A_sd": res["quantities"]["A"]["t"]["sd"] if res["quantities"]["A"]["t"] else None,
            "A_bounds": res["quantities"]["A"].get("worst_case_bounds_on_mean")}


def analytic_power(delta: float, sd: float, n: int, margin: float, alpha: float = 0.05) -> float:
    from scipy import stats
    lam = (delta - margin) * np.sqrt(n) / sd
    return float(stats.nct.sf(stats.t.ppf(1 - alpha, n - 1), n - 1, lam))


def summarize(results: list[dict], n_datasets: int, sd_points: float, config: dict) -> dict:
    out: dict[str, dict] = {}
    for name, truth in SCENARIOS.items():
        rows = [r for r in results if r["scenario"] == name]
        q_probs = {}
        for q in ["A", "O", "B", "V", "O_minus_A", "A_minus_B"]:
            labels = [r["decisions"][q] for r in rows]
            q_probs[q] = {lab: labels.count(lab) / len(rows) for lab in ("supported", "equivalent", "inconclusive")}
        interp = {k: np.mean([any(v in s for s in r["interpretation"]) for r in rows]) for k, v in INTERPRETATION_KEYS.items()}
        a_sd = [r["A_sd"] for r in rows if r["A_sd"] is not None]
        out[name] = {"truth": truth, "n_datasets": len(rows), "decision_probabilities": q_probs, "interpretation_frequency": interp,
                     "realized_paired_sd_A": {"mean": float(np.mean(a_sd)) if a_sd else None}, "mock_first_dataset": rows[0]["decisions"] if rows else None,
                     "A_worst_case_bounds_first_dataset": rows[0]["A_bounds"] if rows else None}
    # validation against the registration's claims
    n = n_datasets
    tol = lambda p: 2.5 * np.sqrt(p * (1 - p) / n)  # noqa: E731
    margin = config["decision"]["margin_points"]
    seeds = len(config["seeds"])
    pw = analytic_power(5.0, sd_points, seeds, margin)
    checks = [
        {"check": "false support of A at the margin (Δ_A = 3) ≤ 5% (+MC tolerance)", "value": out["A_at_margin_3"]["decision_probabilities"]["A"]["supported"], "bound": 0.05 + tol(0.05),
         "pass": out["A_at_margin_3"]["decision_probabilities"]["A"]["supported"] <= 0.05 + tol(0.05)},
        {"check": "false equivalence of A at the lower margin (Δ_A = −3) ≤ 5% (+MC tolerance)", "value": out["A_at_minus_margin_-3"]["decision_probabilities"]["A"]["equivalent"], "bound": 0.05 + tol(0.05),
         "pass": out["A_at_minus_margin_-3"]["decision_probabilities"]["A"]["equivalent"] <= 0.05 + tol(0.05)},
        {"check": f"power for A at Δ_A = 5, SD {sd_points} matches the analytic value {pw:.3f} (±MC tolerance)", "value": out["A_5_planning"]["decision_probabilities"]["A"]["supported"], "bound": pw,
         "pass": abs(out["A_5_planning"]["decision_probabilities"]["A"]["supported"] - pw) <= tol(max(min(pw, 0.95), 0.05)) + 0.02},
        {"check": "missing run makes A, O−A and A−B inconclusive in every dataset", "value": min(out["missing_run_historical_10"]["decision_probabilities"][q]["inconclusive"] for q in ("A", "O_minus_A", "A_minus_B")), "bound": 1.0,
         "pass": all(out["missing_run_historical_10"]["decision_probabilities"][q]["inconclusive"] == 1.0 for q in ("A", "O_minus_A", "A_minus_B"))},
        {"check": "global null: familywise false 'supported' among secondaries ≤ 5% (+MC tolerance)", "value": None, "bound": 0.05 + tol(0.05), "pass": None},
        {"check": "historical 10-point effect: all four drops supported with probability ≥ 0.99", "value": min(out["historical_10"]["decision_probabilities"][q]["supported"] for q in ("A", "O", "B", "V")), "bound": 0.99,
         "pass": all(out["historical_10"]["decision_probabilities"][q]["supported"] >= 0.99 for q in ("A", "O", "B", "V"))},
    ]
    rows = [r for r in results if r["scenario"] == "global_null"]
    fw = np.mean([any(r["decisions"][q] == "supported" for q in config["decision"]["secondary"]) for r in rows])
    checks[4]["value"] = float(fw)
    checks[4]["pass"] = bool(fw <= 0.05 + tol(0.05))
    return {"sd_points": sd_points, "n_datasets": n_datasets, "analytic_power_A5": pw, "scenarios": out, "validation": checks,
            "all_checks_pass": all(c["pass"] for c in checks)}


def table_markdown(summary: dict) -> str:
    L = [f"**Paired SD {summary['sd_points']} points, {summary['n_datasets']} synthetic datasets per truth.** Analytic power at Δ_A = 5: {summary['analytic_power_A5']:.3f}.", "",
         "| Truth (Δ_O, Δ_A, Δ_B, Δ_V in points) | A supported | A equivalent | A inconclusive | O | B | V | O−A | A−B | Interpretation rows (freq.) |", "|---|---|---|---|---|---|---|---|---|---|"]

    def top(d: dict) -> str:
        k = max(d, key=d.get)
        return f"{k} {d[k]:.2f}"

    for name, s in summary["scenarios"].items():
        t = s["truth"]
        extra = ("; missing " + ", ".join(t["missing"])) if t.get("missing") else ""
        extra += f"; SD {t['sd']}" if t.get("sd") else ""
        dp = s["decision_probabilities"]
        interp = "; ".join(f"{k} {v:.2f}" for k, v in s["interpretation_frequency"].items() if v > 0) or "—"
        L.append(f"| {name}: ({t['O']}, {t['A']}, {t['B']}, {t['V']}){extra} | {dp['A']['supported']:.2f} | {dp['A']['equivalent']:.2f} | {dp['A']['inconclusive']:.2f} | "
                 f"{top(dp['O'])} | {top(dp['B'])} | {top(dp['V'])} | {top(dp['O_minus_A'])} | {top(dp['A_minus_B'])} | {interp} |")
    L += ["", "| Validation | Value | Bound | Pass |", "|---|---|---|---|"]
    for c in summary["validation"]:
        L.append(f"| {c['check']} | {c['value']:.3f} | {c['bound']:.3f} | {'yes' if c['pass'] else 'NO'} |")
    return "\n".join(L) + "\n"


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Synthetic dry run of the frozen Experiment 2 analyzer.")
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--selection", required=True)
    ap.add_argument("--support", required=True)
    ap.add_argument("--config", default=None)
    ap.add_argument("--datasets", type=int, default=200)
    ap.add_argument("--sd", type=float, action="append", default=None, help="paired SD in points (repeat); default 1.88 and 3.0")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--out", required=True, help="output JSON; .md written alongside")
    args = ap.parse_args(argv)
    config = load_config(args.config)
    paths = {"config": args.config or str(PACKAGE_ROOT / "configs" / "exp2_registered.json"), "selection": args.selection, "support": args.support, "manifest": args.manifest}
    sds = args.sd or [1.88, 3.0]
    summaries = {}
    md = ["# Experiment 2 dry run: operating characteristics of the frozen analyzer", ""]
    for sd in sds:
        jobs = [(name, truth, sd, k, paths) for name, truth in SCENARIOS.items() for k in range(args.datasets)]
        with ProcessPoolExecutor(args.workers) as ex:
            results = list(ex.map(_one, jobs, chunksize=4))
        summaries[str(sd)] = summarize(results, args.datasets, sd, config)
        md.append(table_markdown(summaries[str(sd)]))
    write_json(Path(args.out), {"scenarios": SCENARIOS, "by_sd": summaries, "config_sha256": sha256_json(config)})
    Path(args.out).with_suffix(".md").write_text("\n".join(md))
    print("\n".join(md))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/freeze.py
"""Freeze manifest for Experiment 2.

The manifest binds, by SHA-256: the registered config, the episode manifest, the certified selection
(all 110 run id lists), the evaluation support and weights, and the analyzer code. It records dated,
named sign-offs, and the environment: the lockfile hash of the owner's project, the pinned runtime
versions with their hash, and the hardware class of the registered runs. ``verify_freeze`` is called
by the driver before training (code, selection and environment block must match) and by the analyzer
before any decision (sign-offs must also be present); ``check_runtime`` compares the live runtime and
device with the frozen ones when training starts. Hashes bind bytes;
the sign-off strings reject placeholders but cannot prove approval, so the manifest should be stored
next to the approval record (for example, the OSF registration and the owner's email).
"""

from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import importlib.metadata
import json
import platform
import re
from pathlib import Path

from . import PACKAGE_ROOT, load_config, sha256_file, sha256_json, write_json

CODE_FILES = ("registered/__init__.py", "registered/manifest.py", "registered/design.py", "registered/weights.py",
              "registered/analyze.py", "registered/freeze.py", "registered/run_registered.py")
REQUIRED_ROLES = ("paper_lead", "stats_check")  # internal signers; the experiment owner may add a third line
PLACEHOLDERS = {"", "name", "full name", "tbd", "todo", "x", "owner", "lead"}
DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
HEX64 = re.compile(r"^[0-9a-f]{64}$")
RUNTIME_PACKAGES = ("torch", "numpy", "scipy", "scikit-learn", "pandas", "transformers")


class FreezeError(Exception):
    pass


def code_hashes(root: Path = PACKAGE_ROOT) -> dict[str, str]:
    return {f: sha256_file(root / f) for f in CODE_FILES if (root / f).exists()}


def runtime_versions() -> dict[str, str]:
    versions = {"python": platform.python_version()}
    for name in RUNTIME_PACKAGES:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "absent"
    try:
        import torch  # noqa: PLC0415  (optional)

        versions["cuda"] = str(torch.version.cuda or "none")
    except ImportError:
        versions["cuda"] = "none"
    return versions


def runtime_sha256(versions: dict[str, str]) -> str:
    return hashlib.sha256(json.dumps(versions, sort_keys=True).encode()).hexdigest()


def detected_hardware(device: str) -> str:
    if str(device).startswith("cuda"):
        import torch  # noqa: PLC0415

        return torch.cuda.get_device_name(0)
    if str(device).startswith("mps"):
        return f"Apple MPS ({platform.machine()})"
    return f"CPU ({platform.processor() or platform.machine()})"


def environment_block(hardware_class: str, lockfile: Path) -> dict:
    """Lockfile hash of the owner's project, pinned runtime versions and their hash, hardware class."""

    versions = runtime_versions()
    return {"lockfile": str(lockfile), "lockfile_sha256": sha256_file(Path(lockfile)), "runtime": versions,
            "runtime_sha256": runtime_sha256(versions), "hardware_class": hardware_class}


def validate_environment(environment: object, lockfile: Path | None = None) -> None:
    if not isinstance(environment, dict):
        raise FreezeError("the freeze manifest has no environment block")
    if not HEX64.match(str(environment.get("lockfile_sha256", ""))):
        raise FreezeError("environment lockfile_sha256 is missing")
    if lockfile is not None and environment["lockfile_sha256"] != sha256_file(Path(lockfile)):
        raise FreezeError("the owner's lockfile differs from the frozen one")
    runtime = environment.get("runtime")
    if not isinstance(runtime, dict) or not {"python", *RUNTIME_PACKAGES} <= set(runtime):
        raise FreezeError("environment runtime must list python and every registered package")
    if environment.get("runtime_sha256") != runtime_sha256(runtime):
        raise FreezeError("environment runtime_sha256 does not match its runtime versions")
    hardware = str(environment.get("hardware_class", "")).strip()
    if len(hardware) < 2 or hardware.lower() in PLACEHOLDERS:
        raise FreezeError("environment must name the hardware class of the registered runs")


def check_runtime(freeze: dict, device: str, deviation: str | None = None) -> dict:
    """Refuse training on another hardware class or runtime unless a logged deviation is given."""

    environment = freeze["environment"]
    live = runtime_versions()
    detected = detected_hardware(device)
    differences = {k: [environment["runtime"].get(k), v] for k, v in live.items() if environment["runtime"].get(k) != v}
    hardware_ok = environment["hardware_class"].strip().lower() in detected.lower()
    record = {"declared_hardware_class": environment["hardware_class"], "detected_hardware": detected, "hardware_matches": hardware_ok,
              "runtime_differences": differences, "runtime_sha256_live": runtime_sha256(live), "deviation": deviation}
    if (differences or not hardware_ok) and not (deviation and len(deviation.strip()) >= 10):
        raise FreezeError(f"environment differs from the freeze: hardware {detected!r} vs {environment['hardware_class']!r}, runtime {differences}; "
                          "rerun on the frozen environment or pass --deviation with a reason (logged)")
    return record


def build_freeze(config: dict, manifest_path: Path, selection: dict, support: dict, approvals: list[dict], registration: str | None,
                 environment: dict | None = None) -> dict:
    if selection.get("manifest_kind") == "SYNTHETIC":
        raise FreezeError("refusing to freeze a selection built on a SYNTHETIC manifest")
    if not selection["certificate"]["feasible"]:
        raise FreezeError("refusing to freeze: certificate reports failed checks")
    if support["manifest_sha256"] != selection["manifest_sha256"]:
        raise FreezeError("support and selection were built from different manifests")
    return {
        "created": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "config_name": config["name"], "config_version": config["version"], "config_sha256": sha256_json(config),
        "manifest_sha256": selection["manifest_sha256"], "manifest_path": str(manifest_path),
        "selection_sha256": selection["selection_sha256_without_certificate"],
        "support_sha256": support["sha256"],
        "code_sha256": code_hashes(),
        "design_B_budget": selection["design_B"]["budget_transitions"], "n_runs": selection["n_runs"],
        "approvals": approvals, "registration": registration, "environment": environment,
        "status": "signed" if {a.get("role") for a in approvals} >= set(REQUIRED_ROLES) else "DRAFT - unsigned",
        "statement": "Training runs start only after this manifest is signed and the registration is submitted. No hypothesis, margin, weight, support, or selection changes afterwards.",
    }


def verify_freeze(freeze: dict, *, config: dict, selection: dict | None = None, support: dict | None = None,
                  require_signoff: bool, check_code: bool = True, lockfile: Path | None = None) -> None:
    if sha256_json(config) != freeze["config_sha256"]:
        raise FreezeError("config differs from the frozen config")
    if selection is not None and selection["selection_sha256_without_certificate"] != freeze["selection_sha256"]:
        raise FreezeError("selection differs from the frozen selection")
    if support is not None and support["sha256"] != freeze["support_sha256"]:
        raise FreezeError("support/weights differ from the frozen support")
    if check_code:
        now = code_hashes()
        changed = [f for f, h in freeze["code_sha256"].items() if now.get(f) != h]
        if changed:
            raise FreezeError(f"code changed since freeze: {changed}")
    validate_environment(freeze.get("environment"), lockfile)
    if require_signoff:
        roles = {a.get("role") for a in freeze.get("approvals", [])}
        missing = [r for r in REQUIRED_ROLES if r not in roles]
        if missing:
            raise FreezeError(f"missing sign-off roles: {missing}")
        for a in freeze["approvals"]:
            if str(a.get("name", "")).strip().lower() in PLACEHOLDERS or not DATE.match(str(a.get("date", ""))):
                raise FreezeError(f"placeholder or undated approval: {a}")


def sign(freeze: dict, role: str, name: str, date: str) -> dict:
    """Add one internal sign-off to a draft; it becomes signed once every required role has signed."""

    if role not in REQUIRED_ROLES:
        raise FreezeError(f"role must be one of {REQUIRED_ROLES}")
    if str(name).strip().lower() in PLACEHOLDERS or len(str(name).split()) < 2:
        raise FreezeError("sign with a full name")
    if not DATE.match(date):
        raise FreezeError("date must be YYYY-MM-DD")
    approvals = [a for a in freeze.get("approvals", []) if a.get("role") != role] + [{"role": role, "name": name, "date": date}]
    signed = {**freeze, "approvals": approvals}
    signed["status"] = "signed" if {a["role"] for a in approvals} >= set(REQUIRED_ROLES) else "DRAFT - unsigned"
    return signed


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Create or verify the Experiment 2 freeze manifest.")
    sub = ap.add_subparsers(dest="cmd", required=True)
    c = sub.add_parser("create")
    c.add_argument("--config", default=None)
    c.add_argument("--manifest", required=True)
    c.add_argument("--selection", required=True)
    c.add_argument("--support", required=True)
    c.add_argument("--approve", action="append", default=[], help='"role:Full Name:YYYY-MM-DD" (repeat)')
    c.add_argument("--registration", default=None, help="OSF registration URL or DOI")
    c.add_argument("--hardware-class", required=True, help='hardware of the registered runs, for example "A100" or "Apple M"')
    c.add_argument("--lockfile", required=True, help="uv.lock of the owner's project (robot-world-model-optimism-research)")
    c.add_argument("--out", required=True)
    g = sub.add_parser("sign", help="add one internal sign-off to a freeze manifest")
    g.add_argument("--freeze", required=True)
    g.add_argument("--role", required=True, choices=REQUIRED_ROLES)
    g.add_argument("--name", required=True)
    g.add_argument("--date", required=True, help="YYYY-MM-DD")
    v = sub.add_parser("verify")
    v.add_argument("--freeze", required=True)
    v.add_argument("--config", default=None)
    v.add_argument("--selection", default=None)
    v.add_argument("--support", default=None)
    v.add_argument("--require-signoff", action="store_true")
    v.add_argument("--lockfile", default=None, help="check the owner's uv.lock against the frozen hash")
    args = ap.parse_args(argv)
    config = load_config(getattr(args, "config", None))
    if args.cmd == "create":
        approvals = []
        for a in args.approve:
            role, name, date = a.split(":", 2)
            approvals.append({"role": role, "name": name, "date": date})
        fz = build_freeze(config, Path(args.manifest), json.loads(Path(args.selection).read_text()), json.loads(Path(args.support).read_text()), approvals, args.registration,
                          environment_block(args.hardware_class, Path(args.lockfile)))
        write_json(Path(args.out), fz)
        print(json.dumps({k: v for k, v in fz.items() if k != "code_sha256"}, indent=1))
    elif args.cmd == "sign":
        signed = sign(json.loads(Path(args.freeze).read_text()), args.role, args.name, args.date)
        write_json(Path(args.freeze), signed)
        print(f"{args.freeze}: {signed['status']}; signed by {sorted(a['role'] for a in signed['approvals'])}")
    else:
        fz = json.loads(Path(args.freeze).read_text())
        verify_freeze(fz, config=config, selection=json.loads(Path(args.selection).read_text()) if args.selection else None,
                      support=json.loads(Path(args.support).read_text()) if args.support else None, require_signoff=args.require_signoff,
                      lockfile=Path(args.lockfile) if args.lockfile else None)
        print("freeze manifest verified")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/exp2-kuka-matched/registered/run_registered.py
"""Registered training driver for Experiment 2.

Replaces ``run_arms.py`` for the registered runs. Differences that matter for the registration:

* training sets come from the frozen selection (ids by hash), never from episode counts chosen here;
* the freeze manifest must verify (code, config, selection, support unchanged; sign-offs present)
  before any training starts, unless ``--smoke`` runs the synthetic smoke test;
* normalization statistics and the judge are fitted once from frozen, hash-recorded episode sets; the
  judge validity gate (real AUROC >= 0.75) is a hard stop, not a warning;
* every run writes per-episode scores, the training ids hash, the recomputed distinct-transition
  count, the optimizer exposure (steps x batch), a checkpoint hash and the config and selection
  hashes; an existing run file with different hashes is refused (no silent reuse of stale artefacts);
* an infrastructure failure writes a run file with ``status: infrastructure_failure`` and the
  traceback; ``--retry-failed`` reruns such runs once with the same seed, as registered.

The model, trainer, rollout and judge are the owner's unchanged ``wm.py`` (``--owner-repo``).
"""

from __future__ import annotations

import argparse
import datetime as dt
import glob
import hashlib
import json
import pickle
import sys
import time
import traceback
from pathlib import Path

import numpy as np

from . import load_config, sha256_json, write_json
from .manifest import Manifest, load_manifest


def import_owner_wm(owner_repo: Path):
    sys.path.insert(0, str(Path(owner_repo) / "experiment"))
    import wm  # type: ignore

    return wm


# ---------------------------------------------------------------- loading a frozen subset

def load_subset(wm, npz_dir: Path, m: Manifest, ids: list[str]):
    """Load features/actions for exactly ``ids`` (manifest order) into the owner's ``Episodes``."""

    idx = m.index_of()
    rows = np.array(sorted(idx[i] for i in ids), int)
    by_shard: dict[int, list[int]] = {}
    for r in rows:
        by_shard.setdefault(int(m.shard[r]), []).append(int(r))
    n_frames = int(m.length[rows].sum())
    Z = np.empty((n_frames, 17, 384), np.float16)
    A = None
    L, S, sh, out_ids = [], [], [], []
    row = 0
    for sid in sorted(by_shard):
        f = Path(npz_dir) / f"shard_{sid:05d}.npz"
        with np.load(f) as npz:
            feats, acts, succ, ln = npz["feats"], npz["actions"], npz["success"], npz["length"]
        if A is None:
            A = np.empty((n_frames, acts.shape[-1]), np.float32)
        for r in by_shard[sid]:
            e = int(m.ids[r].split("/")[1])
            n = int(m.length[r])
            if int(ln[e]) != n or int(succ[e]) != int(m.success[r]):
                raise RuntimeError(f"manifest/cache mismatch for {m.ids[r]}: cache length {ln[e]} success {succ[e]}")
            Z[row:row + n], A[row:row + n] = feats[e, :n], acts[e, :n]
            row += n
            L.append(n); S.append(int(succ[e])); sh.append(sid); out_ids.append(str(m.ids[r]))
        del feats, acts
    assert row == n_frames
    ep = wm.Episodes(Z, A, np.array(L), np.array(S), np.array(sh))
    ep.ids = np.array(out_ids, dtype=object)
    ep.pos = {i: k for k, i in enumerate(out_ids)}
    return ep


def rows_for(ep, ids: list[str]) -> np.ndarray:
    return np.array([ep.pos[i] for i in ids], int)


def sha256_path(p: Path) -> str:
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


# ---------------------------------------------------------------- judge

def fit_or_load_judge(wm, ep, m: Manifest, config: dict, out: Path, train_ids: list[str], test_ids: list[str], log) -> tuple[object, dict]:
    jc = config["judge"]
    jpath, jmeta = out / "judge.pkl", out / "judge.json"
    y_train = np.array([m.success[m.index_of()[i]] for i in train_ids])
    rng = np.random.default_rng(jc["rng_seed"])
    pool_ids = np.array(train_ids, dtype=object)
    j_ids = np.concatenate([rng.permutation(pool_ids[y_train == c])[: jc["per_class"]] for c in (0, 1)]).tolist()
    expected = {"judge_ids_sha256": sha256_json(sorted(j_ids)), "config_sha256": sha256_json(config)}
    if jmeta.exists():
        meta = json.loads(jmeta.read_text())
        if {k: meta.get(k) for k in expected} != expected or not jpath.exists() or sha256_path(jpath) != meta["judge_pkl_sha256"]:
            raise RuntimeError("existing judge.json/judge.pkl do not match the frozen configuration; refusing to reuse")
        log("judge: loaded frozen judge")
        return pickle.loads(jpath.read_bytes()), meta
    log(f"judge: fitting on {len(j_ids)} real train trajectories")
    j_rows = rows_for(ep, j_ids)
    judge = wm.SuccessJudge(C=jc["C"]).fit(wm.real_summary(ep, j_rows), ep.success[j_rows])
    te_rows = rows_for(ep, test_ids)
    p_real = judge.prob(wm.real_summary(ep, te_rows))
    y_te = ep.success[te_rows]
    from sklearn.metrics import roc_auc_score
    h = p_real > config["evaluation"]["threshold"]
    bins = np.clip((p_real * 10).astype(int), 0, 9)
    reliability = [{"bin": int(b), "n": int((bins == b).sum()), "mean_p": float(p_real[bins == b].mean()) if (bins == b).any() else None,
                    "frac_success": float(y_te[bins == b].mean()) if (bins == b).any() else None} for b in range(10)]
    meta = {**expected, "eval_ids": test_ids, "p_real": p_real.tolist(), "auroc": float(roc_auc_score(y_te, p_real)),
            "pred_rate_unweighted": float(h.mean()), "true_rate_unweighted": float(y_te.mean()),
            "false_positive_rate": float(h[y_te == 0].mean()), "false_negative_rate": float((~h[y_te == 1]).mean()),
            "brier_unweighted": float(((p_real - y_te) ** 2).mean()), "reliability": reliability,
            "length_only_auroc": float(roc_auc_score(y_te, -ep.L[te_rows])), "validity_gate": jc["validity_auroc"]}
    jpath.write_bytes(pickle.dumps(judge))
    meta["judge_pkl_sha256"] = sha256_path(jpath)
    write_json(jmeta, meta)
    log(f"judge: real AUROC {meta['auroc']:.3f} (gate {jc['validity_auroc']}), FPR {meta['false_positive_rate']:.3f}, FNR {meta['false_negative_rate']:.3f}")
    return judge, meta


# ---------------------------------------------------------------- runs

def run_one(wm, ep, judge, run: dict, config: dict, selection: dict, test_ids: list[str], out: Path, device_note: str, log, attempt: int) -> dict:
    rows = rows_for(ep, run["ids"])
    ids_hash = sha256_json(sorted(run["ids"]))
    if ids_hash != run["ids_sha256"]:
        raise RuntimeError(f"{run['run_id']}: training ids hash differs from the frozen selection")
    rec = {"run_id": run["run_id"], "design": run["design"], "arm": run["arm"], "phi": run["phi"], "seed": run["seed"],
           "selection_sha256": selection["selection_sha256_without_certificate"], "config_sha256": sha256_json(config), "ids_sha256": ids_hash,
           "attempt": attempt, "started": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "device": device_note,
           "train": {"n_episodes": int(len(rows)), "n_success": int(ep.success[rows].sum()), "n_failure": int((1 - ep.success[rows]).sum()),
                     "distinct_transitions": int((ep.L[rows] - 1).sum()), "optimizer_steps": config["optimizer_steps"], "batch_size": 256,
                     "sampled_starts": config["optimizer_steps"] * 256, "rollout_horizon": 4,
                     "note": "valid multistep loss terms are not exposed by wm.train_wm; add a counter there to log them"}}
    t0 = time.time()
    try:
        model = wm.train_wm(ep, rows, config["optimizer_steps"], seed=run["seed"], log=log)
        te_rows = rows_for(ep, test_ids)
        summ, _ = wm.predict_summary(model, ep, te_rows)
        p = judge.prob(summ)
        import torch
        ckpt = out / "checkpoints" / f"{run['run_id']}.pt"
        ckpt.parent.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), ckpt)
        rec.update(status="completed", eval_ids=test_ids, p=np.clip(p, 0, 1).tolist(), checkpoint=str(ckpt), checkpoint_sha256=sha256_path(ckpt),
                   train_minutes=(time.time() - t0) / 60, finished=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"))
    except Exception:  # noqa: BLE001  infrastructure failure: record, do not analyse around it
        rec.update(status="infrastructure_failure", error=traceback.format_exc(), train_minutes=(time.time() - t0) / 60)
    return rec


def run_all(*, owner_repo: Path, npz_dir: Path, manifest_path: Path, selection_path: Path, support_path: Path, config_path: Path | None,
            out: Path, freeze_path: Path | None, smoke: bool, only: list[str] | None, retry_failed: bool, allow_judge_below_gate: bool,
            deviation: str | None = None,
            log=print) -> list[dict]:
    wm = import_owner_wm(owner_repo)
    config = load_config(config_path)
    selection = json.loads(Path(selection_path).read_text())
    support = json.loads(Path(support_path).read_text())
    m, mhash = load_manifest(manifest_path)
    if mhash != selection["manifest_sha256"] or mhash != support["manifest_sha256"]:
        raise RuntimeError("manifest hash differs from the one the selection/support were built from")
    if not selection["certificate"]["feasible"]:
        raise RuntimeError("selection certificate failed; refusing to train")
    if not smoke:
        if freeze_path is None:
            raise RuntimeError("a signed freeze manifest is required to start registered training (use --smoke for the synthetic smoke test)")
        from .freeze import check_runtime, verify_freeze
        frozen = json.loads(Path(freeze_path).read_text())
        verify_freeze(frozen, config=config, selection=selection, support=support, require_signoff=True,
                      lockfile=Path(owner_repo) / "uv.lock")
        environment_check = check_runtime(frozen, wm.DEVICE, deviation)
        out.mkdir(parents=True, exist_ok=True)
        write_json(out / f"environment_check_{dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json", environment_check)
        log(f"environment check: {json.dumps(environment_check)}")
        if selection.get("manifest_kind") == "SYNTHETIC":
            raise RuntimeError("SYNTHETIC manifest; refusing to train")
    out.mkdir(parents=True, exist_ok=True)
    runs = selection["runs"] if not only else [r for r in selection["runs"] if r["run_id"] in set(only)]

    train_ids = sorted(m.ids[m.train_mask()].tolist())
    test_ids = sorted(m.ids[m.test_mask()].tolist())
    needed = set(test_ids) | set(train_ids)  # judge pool and normalization draw from the full train pool
    t0 = time.time()
    ep = load_subset(wm, npz_dir, m, sorted(needed))
    log(f"loaded {len(ep)} episodes ({ep.Z.shape[0]} frames) in {time.time() - t0:.0f}s; peak RSS {wm.rss_gb():.2f} GB")
    norm_ids = train_ids[:2000]  # frozen: first 2,000 eligible train episodes in canonical manifest order
    norm_rows = rows_for(ep, norm_ids)
    stat_rows = np.concatenate([np.arange(ep.off[e], ep.off[e] + ep.L[e]) for e in norm_rows])
    wm.normalize_inplace(ep, stat_rows)
    write_json(out / "normalization.json", {"normalization_ids_sha256": sha256_json(norm_ids), "n_episodes": len(norm_ids), "n_frames": int(len(stat_rows))})
    judge, jmeta = fit_or_load_judge(wm, ep, m, config, out, train_ids, test_ids, log)
    if jmeta["auroc"] < config["judge"]["validity_auroc"] and not allow_judge_below_gate:
        raise RuntimeError(f"judge validity gate failed: real AUROC {jmeta['auroc']:.3f} < {config['judge']['validity_auroc']}; registered runs may not start")
    device_note = f"torch device {wm.DEVICE}"
    results = []
    for run in runs:
        path = out / f"{run['run_id']}.json"
        attempt = 1
        if path.exists():
            prev = json.loads(path.read_text())
            if prev.get("selection_sha256") != selection["selection_sha256_without_certificate"] or prev.get("config_sha256") != sha256_json(config):
                raise RuntimeError(f"{run['run_id']}: existing run file was produced under a different selection/config; refusing to reuse or overwrite")
            if prev.get("status") == "completed" or not (retry_failed and prev.get("status") == "infrastructure_failure" and prev.get("attempt", 1) < 2):
                results.append(prev)
                continue
            attempt = prev.get("attempt", 1) + 1
            log(f"{run['run_id']}: retrying once with the same seed (attempt {attempt})")
        rec = run_one(wm, ep, judge, run, config, selection, test_ids, out, device_note, log, attempt)
        write_json(path, rec)
        results.append(rec)
        if rec["status"] == "completed":
            w = np.array([support["weights"].get(i, 0.0) for i in test_ids])
            log(f"{run['run_id']}: weighted thresholded rate {(w * (np.array(rec['p']) > 0.5)).sum():.4f}  [{rec['train_minutes']:.1f} min, peak RSS {wm.rss_gb():.2f} GB]")
        else:
            log(f"{run['run_id']}: INFRASTRUCTURE FAILURE (attempt {attempt}); see run file")
    return results


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Train the registered Experiment 2 runs from the frozen selection.")
    ap.add_argument("--owner-repo", required=True, help="path to robot-world-model-optimism-research (uses experiment/wm.py)")
    ap.add_argument("--npz-dir", required=True, help="data/kuka with shard_*.npz")
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--selection", required=True)
    ap.add_argument("--support", required=True)
    ap.add_argument("--config", default=None)
    ap.add_argument("--freeze", default=None, help="signed freeze manifest (required unless --smoke)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--only", nargs="*", default=None, help="run ids to train (default: all)")
    ap.add_argument("--retry-failed", action="store_true", help="rerun infrastructure failures once with the same seed")
    ap.add_argument("--allow-judge-below-gate", action="store_true", help="deviation: continue although the judge gate failed (recorded)")
    ap.add_argument("--smoke", action="store_true", help="synthetic smoke test: skips the freeze requirement")
    ap.add_argument("--deviation", default=None, help="reason to train on another hardware class or runtime than the freeze (logged)")
    args = ap.parse_args(argv)
    run_all(owner_repo=Path(args.owner_repo), npz_dir=Path(args.npz_dir), manifest_path=Path(args.manifest), selection_path=Path(args.selection),
            support_path=Path(args.support), config_path=args.config, out=Path(args.out), freeze_path=Path(args.freeze) if args.freeze else None,
            smoke=args.smoke, only=args.only, retry_failed=args.retry_failed, allow_judge_below_gate=args.allow_judge_below_gate,
            deviation=args.deviation)


if __name__ == "__main__":
    main()


In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title B4. Manifest → selection + certificate → support → dry run (CPU, 5–15 min)
%cd /content/exp2-kuka-matched
import os
!PYTHONPATH=. python -m registered.manifest build --npz-dir /content/rwmo/data/kuka --out {OUT}/manifest.jsonl
!PYTHONPATH=. python -m registered.design --manifest {OUT}/manifest.jsonl --out {OUT}/selection.json | head -n 60
!PYTHONPATH=. python -m registered.weights --manifest {OUT}/manifest.jsonl --out {OUT}/support.json | tail -n 30
!PYTHONPATH=. python -m registered.dryrun --manifest {OUT}/manifest.jsonl --selection {OUT}/selection.json --support {OUT}/support.json --datasets {DRYRUN_DATASETS} --workers 2 --out {OUT}/dryrun.json | tail -n 40
import json
c = json.load(open(f"{OUT}/selection.certificate.json"))
print(json.dumps({k: v for k, v in c.items() if k not in ("checks", "failed")}, indent=1)); print("failed checks:", c["failed"][:5])

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title B5. Package Exp 2 results for download
!cd {OUT} && zip -q -j /content/exp2_prep_results.zip manifest.summary.json selection.json selection.certificate.json selection.md support.json support.md dryrun.json dryrun.md
!ls -la /content/exp2_prep_results.zip
from google.colab import files
files.download("/content/exp2_prep_results.zip")

## Part A — finish: wait for the pilot, summarize, download

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title A4. Wait for the pilot (prints progress lines as they appear)
import subprocess, time, os
def pilot_running():
    return subprocess.run(["pgrep", "-f", "exposure_experiment.py --config experiments/exposure2d_capacity_pilot_v1.json"], capture_output=True).returncode == 0
seen = 0
while pilot_running():
    lines = open(EXP3_LOG).read().splitlines() if os.path.exists(EXP3_LOG) else []
    for line in lines[seen:]:
        if line.startswith(("block", "resuming")):
            print(line, flush=True)
    seen = len(lines)
    time.sleep(60)
print("pilot process finished")
!tail -n 5 {EXP3_LOG}
!cp /content/trajectory-lineage/results/exposure2d_capacity_pilot_v1_blinded.json {EXP3_DRIVE_DIR}/ 2>/dev/null || echo "no report yet: if blocks are missing, Runtime -> Run all resumes from the checkpoint" 

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title A5. Blinded summary and download (dispersions, competence, runtime per size; pooled positive-control flag)
import json
path = f"{EXP3_DRIVE_DIR}/exposure2d_capacity_pilot_v1_blinded.json"
r = json.load(open(path))
print({k: r[k] for k in ("blocks_scheduled", "blocks_completed", "blocks_base_competent", "positive_control_passed") if k in r})
for level, v in r["by_capacity"].items():
    disp = {name: round(d["standard_deviation_upper_limit"], 5) for name, d in v["dispersion"].items()}
    print(f"hidden {level}: params {v['model_parameters']}, completed {v['blocks_completed']}, competent {v['blocks_base_competent']}, s/block {None if v['seconds_per_block_mean'] is None else round(v['seconds_per_block_mean'])}, SD upper limits {disp}")
print("infrastructure failures:", len(r["infrastructure_failures"]), "| algorithmic failures:", len(r["algorithmic_failures"]))
!git -C /content/trajectory-lineage rev-parse HEAD > {EXP3_DRIVE_DIR}/code_commit.txt
!cd {EXP3_DRIVE_DIR} && zip -q -j /content/exp3_capacity_pilot_results.zip exposure2d_capacity_pilot_v1_blinded.json exposure2d_capacity_base_tuning.json code_commit.txt pilot.log
!ls -la /content/exp3_capacity_pilot_results.zip
from google.colab import files
files.download("/content/exp3_capacity_pilot_results.zip")

---
## Locked — Exp 2 registered training (do not run before the freeze is signed and the registration is submitted)
Upload the signed `freeze.json` to `OUT` first. The driver verifies config, selection, support, code hashes and sign-offs, then trains the runs with the owner's `wm.py` and resumes if the runtime restarts.

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title L1. Train (requires {OUT}/freeze.json)
import os
assert os.path.exists(f"{OUT}/freeze.json"), "no signed freeze manifest in OUT; do not train before registration"
%cd /content/exp2-kuka-matched
!PYTHONPATH=. python -m registered.run_registered --owner-repo /content/rwmo --npz-dir /content/rwmo/data/kuka --manifest {OUT}/manifest.jsonl --selection {OUT}/selection.json --support {OUT}/support.json --freeze {OUT}/freeze.json --out {OUT}/runs 2>&1 | grep -v -i "warn" | tail -n 150

In [ ]:
assert "RUN_PILOT" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title L2. Analyze (after all runs)
%cd /content/exp2-kuka-matched
!PYTHONPATH=. python -m registered.analyze --runs {OUT}/runs --selection {OUT}/selection.json --support {OUT}/support.json --manifest {OUT}/manifest.jsonl --judge {OUT}/runs/judge.json --freeze {OUT}/freeze.json --out {OUT}/decision.json
!cd {OUT} && zip -q -j /content/exp2_decision.zip decision.json decision.md runs/judge.json && ls -la /content/exp2_decision.zip
from google.colab import files
files.download("/content/exp2_decision.zip")

### Notes
- Blinding: the pilot's Drive checkpoint holds blinded records only (contrasts shifted by the key in `DO_NOT_OPEN/`); the report contains dispersions and the pooled positive-control flag, no effect locations. Never open the key folder. Do not add `--unblind` or `--checkpoint` to the pilot command.
- Resume: after any disconnect, Runtime → Run all. Part A skips finished blocks, Part B skips finished shards and existing outputs.